In [9]:
# ============================================================
# gbpusd_smc_model.py  —  GBP/USD  |  Smart Money Concepts (SMC) School — GBP/USD Forex

# ============================================================

import os
import json
import time
import logging
import warnings
from datetime import datetime, timedelta

import requests
import numpy as np
import pandas as pd
import xgboost as xgb
import joblib
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score

warnings.filterwarnings("ignore")
np.random.seed(42)

# ══════════════════════════════════════════════════════════════
# CONFIG
# ══════════════════════════════════════════════════════════════


TWELVE_DATA_API_KEY = os.environ.get("TWELVE_DATA_API_KEY", "")
if not TWELVE_DATA_API_KEY:
    raise RuntimeError(
        "TWELVE_DATA_API_KEY غير موجود. اضبطه كمتغير بيئة قبل التشغيل ولا "
        "تكتبه أبداً داخل الكود (المفتاح القديم كان hardcoded ولازم يتلغى "
        "من TwelveData ويتستبدل بمفتاح جديد)."
    )

SYMBOL      = "GBP/USD"
SYMBOL_SAFE = "GBPUSD"
MODELS_DIR  = "models/smc_gbpusd_v3"
DATA_DIR    = "data/smc_gbpusd_v3"
HOLDOUT_PCT = 0.15   # آخر 15% من كل فريم — معزولة تماماً عن التدريب

# ─── مواصفات تنفيذ خاصة بـ GBP/USD ───
# GBP/USD سبريده النسبي (كنسبة من السعر) أعلى من الدهب عادة — سيولة أقل
# وحجم تداول أصغر عالمياً، فبنفترض سبريد وسليباج أعلى في الباك-تيست
# عشان الأرقام متبقاش متفائلة زيادة عن الواقع.
INSTRUMENT_SPREAD_BPS = 1.5     # GBP/USD — Cable سبريده أوسع شوية من EUR/USD (~1.5 pips) لكن لسه ضيق كزوج رئيسي      # الدهب كان 2 — الفضة أعرض نسبياً
INSTRUMENT_SLIPPAGE   = 0.00005  # سليباج منخفض — GBP/USD سيولته عالية في جلسة لندن تحديداً  # الدهب كان 0.0001


TIMEFRAME_CONFIG = {
    # ═══ 5min V3: نفس V2 (تحسن 1.64→1.74) ═══
    "5min": {
        "interval": "5min", "years": 3,
        "buy_threshold": 0.64, "sell_threshold": 0.36,
        "fold_buy_threshold": 0.59, "fold_sell_threshold": 0.41,
        "max_trades_per_day": 9,
        "larger_tf": "1h",
        "horizon": 10, "swing_lookback": 5, "target_atr_mult": 0.55,
        "min_candles": 1500, "confidence_threshold": 0.62,
        "atr_sl_mult": 1.3, "atr_tp_mult": 2.2,
    },
    # ═══ 15min V3: نفس V2 (الأفضل، تحسن 2.31→2.68) ═══
    "15min": {
        "interval": "15min", "years": 4,
        "buy_threshold": 0.65, "sell_threshold": 0.35,
        "fold_buy_threshold": 0.60, "fold_sell_threshold": 0.40,
        "max_trades_per_day": 6,
        "larger_tf": "4h",
        "horizon": 10, "swing_lookback": 5, "target_atr_mult": 0.55,
        "min_candles": 1000, "confidence_threshold": 0.62,
        "atr_sl_mult": 1.4, "atr_tp_mult": 2.4,
    },
    # ═══ 45min V3: **استعادة قيم V1 بالضبط** (كان 2.30 وانهار لـ 0.94 في V2) ═══
    # V1 قيمه أعطت أفضل أداء (PF=2.30, 51 صفقة). في V2 قلت "لا تغيير" لكن
    # الأداء انهار — دليل على وجود تفاعل غير مقصود مع تعديلات فريمات تانية.
    # الحل: نعيد نفس قيم V1 بالضبط ونتأمل أن يستعيد أداءه.
    "45min": {
        "interval": "45min", "years": 5,
        "buy_threshold": 0.58, "sell_threshold": 0.37,
        "fold_buy_threshold": 0.56, "fold_sell_threshold": 0.42,
        "max_trades_per_day": 5,
        "larger_tf": "4h",
        "horizon": 8, "swing_lookback": 4, "target_atr_mult": 0.55,
        "min_candles": 700, "confidence_threshold": 0.64,
        "atr_sl_mult": 1.4, "atr_tp_mult": 2.5,
    },
    # ═══ 1h V3: تشديد مضاعف (V2 التشديد اتجاوز — 512 صفقة!) ═══
    # V2 config حط buy/sell=0.65/0.35 لكن الأداء لسه فتح 512 صفقة
    # → معنى: calibration search بيتجاوز buy/sell من config
    # الحل: رفع min_reliable_trades كثيراً + calib_percentile أعلى + max_trades=1
    "1h": {
        "interval": "1h", "years": 6,
        "buy_threshold": 0.58, "sell_threshold": 0.30,        # ★ V3 gap=0.40
        "fold_buy_threshold": 0.56, "fold_sell_threshold": 0.37,
        "max_trades_per_day": 1,                              # ★ V3 (من 2)
        "larger_tf": "4h",
        "horizon": 14, "swing_lookback": 4, "target_atr_mult": 0.55,
        "min_candles": 700, "confidence_threshold": 0.75,     # ★ V3 (من 0.70)
        "atr_sl_mult": 1.3, "atr_tp_mult": 2.0,
        "calib_percentile": 92,                               # ★ V3 (من 85)
        "min_reliable_trades": 40,                            # ★ V3 (من 20) — قوي جداً
    },
    # ═══ 2h V3: تخفيف نسبي (V2 كان تشديد زايد — 9 صفقات فقط) ═══
    # وسط بين V1 (0.58/0.42) و V2 المفرط (0.65/0.35)
    "2h": {
        "interval": "2h", "years": 7,
        "buy_threshold": 0.57, "sell_threshold": 0.38,        # ★ V3 gap=0.24
        "fold_buy_threshold": 0.55, "fold_sell_threshold": 0.42,
        "max_trades_per_day": 3,                              # ★ V3 (من 2)
        "larger_tf": "1d",
        "horizon": 14, "swing_lookback": 4, "target_atr_mult": 0.45,
        "min_candles": 500, "confidence_threshold": 0.68,     # ★ V3 (من 0.72)
        "atr_sl_mult": 1.35, "atr_tp_mult": 1.95,
        "calib_percentile": 80, "cv_splits": 4,
    },
    # ═══ 4h V3: نفس V2 (تحسن 1.12→1.29) ═══
    "4h": {
        "interval": "4h", "years": 6,
        "buy_threshold": 0.57, "sell_threshold": 0.41,
        "fold_buy_threshold": 0.55, "fold_sell_threshold": 0.45,
        "max_trades_per_day": 2,
        "larger_tf": "1d",
        "horizon": 14, "swing_lookback": 3, "target_atr_mult": 0.4,
        "min_candles": 400, "confidence_threshold": 0.68,
        "atr_sl_mult": 1.4, "atr_tp_mult": 2.0, "calib_percentile": 78,
        "cv_splits": 4, "force_resample_source": "1h",
    },
    # ═══ 1d V3: تشديد إضافي (V2 حسّن الأداء بس لسه PF<1) ═══
    # V2 خفف من PF=0.95→0.98 بس لسه خاسر
    # نجرب تشديد أعلى بس نحافظ على SL أوسع
    "1d": {
        "interval": "1day", "years": 12,
        "buy_threshold": 0.55, "sell_threshold": 0.37,        # ★ V3 (من 0.60/0.40)
        "fold_buy_threshold": 0.53, "fold_sell_threshold": 0.42,
        "max_trades_per_day": 1,
        "larger_tf": "1W",
        "horizon": 16, "swing_lookback": 3, "target_atr_mult": 0.55,
        "min_candles": 300, "confidence_threshold": 0.68,     # ★ V3 (من 0.65)
        "atr_sl_mult": 1.6, "atr_tp_mult": 2.1,               # ★ V3 TP أوسع شوية
        "calib_percentile": 88,                               # ★ V3 (من 85)
        "min_reliable_trades": 15, "cv_splits": 3,            # ★ V3
    },
    # ═══ 1W V3: نفس V2 (طبيعي أن العينة صغيرة) ═══
    # V2 أعطى PF=4.18 على 9 صفقات — غير موثوق إحصائياً
    # مفيش حل كودي حقيقي — المشكلة كمية البيانات المتاحة
    "1W": {
        "interval": "1week", "years": 20,
        "buy_threshold": 0.53, "sell_threshold": 0.45,
        "fold_buy_threshold": 0.52, "fold_sell_threshold": 0.48,
        "max_trades_per_day": 1,
        "larger_tf": None,
        "horizon": 10, "swing_lookback": 2, "target_atr_mult": 0.30,
        "min_candles": 80, "confidence_threshold": 0.53,
        "atr_sl_mult": 1.5, "atr_tp_mult": 2.0,
        "calib_percentile": 75,
        "min_reliable_trades": 5, "cv_splits": 3,
    },
}
# قيمة افتراضية للفريمات اللي معندهاش min_reliable_trades محدد صراحة
for _tf_cfg in TIMEFRAME_CONFIG.values():
    _tf_cfg.setdefault("min_reliable_trades", 15)
    _tf_cfg.setdefault("calib_percentile", 88)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    handlers=[logging.FileHandler("training_smc_gbpusd_v3.log"), logging.StreamHandler()],
)

# فحص فوري: كل larger_tf لازم يكون مفتاح موجود فعلاً في TIMEFRAME_CONFIG
# (وليس الـ interval string بتاع الـ API) — يمنع تكرار باگ "تعذر جلب HTF"
for _tf, _cfg in TIMEFRAME_CONFIG.items():
    _ltf = _cfg.get("larger_tf")
    if _ltf is not None and _ltf not in TIMEFRAME_CONFIG:
        raise ValueError(
            f"TIMEFRAME_CONFIG['{_tf}']['larger_tf'] = '{_ltf}' غير موجود كمفتاح "
            f"في TIMEFRAME_CONFIG. المفاتيح المتاحة: {list(TIMEFRAME_CONFIG.keys())}"
        )

# ══════════════════════════════════════════════════════════════
# DATA FETCHER — سحب مباشر لكل فريم بالـ interval الأصلي بتاعه
# ══════════════════════════════════════════════════════════════
class TwelveDataFetcher:
    BASE_URL = "https://api.twelvedata.com/time_series"

    def __init__(self, api_key):
        self.api_key = api_key

    def fetch_range(self, symbol, interval, start_date=None, end_date=None, outputsize=5000):
        params = {
            "symbol": symbol, "interval": interval,
            "apikey": self.api_key, "format": "JSON",
            "order": "ASC", "outputsize": outputsize,
        }
        if start_date:
            params["start_date"] = start_date
        if end_date:
            params["end_date"] = end_date
        try:
            r = requests.get(self.BASE_URL, params=params, timeout=30)
            if r.status_code != 200:
                # نتأكد الأول لو ده فعلاً "وصلنا لأقدم بيانات متاحة" (بترجع
                # عادة كـ HTTP 400 مباشرة مش status 200 + رسالة جوه الـ body
                # زي ما كنا مفترضين قبل كده) — دي حالة طبيعية مش خطأ فعلي.
                try:
                    body = r.json()
                except Exception:
                    body = {}
                body_msg = body.get("message", "")
                if r.status_code == 400 and "No data is available" in body_msg:
                    logging.info(f"[{interval}] وصلنا لأقدم بيانات متاحة على الخطة الحالية.")
                elif r.status_code == 404:
                    logging.info(f"[{interval}] 404 من الـ API — الفريم/التركيبة دي على الأغلب "
                                 f"مش مدعومة مباشرة على الخطة الحالية، هنعتمد على resample.")
                else:
                    logging.error(f"خطأ اتصال {r.status_code}: {r.text[:200]}")
                return pd.DataFrame()
            data = r.json()
            if "values" not in data:
                msg = data.get("message", str(data))
                if data.get("code") == 400 and "No data is available" in msg:
                    # مش خطأ فعلي — وصلنا لحد أقدم بيانات متاحة على الخطة
                    # الحالية في TwelveData لهذا الـ interval. نوقف بهدوء.
                    logging.info(f"[{interval}] وصلنا لأقدم بيانات متاحة على الخطة الحالية.")
                else:
                    logging.error(f"لا بيانات ({interval}): {msg}")
                return pd.DataFrame()
            df = pd.DataFrame(data["values"])
            if "volume" not in df.columns:
                df["volume"] = 0
            for col in ["open", "high", "low", "close", "volume"]:
                df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)
            df["datetime"] = pd.to_datetime(df["datetime"])
            df = df.sort_values("datetime").reset_index(drop=True)
            return df
        except Exception as e:
            logging.error(f"استثناء أثناء الجلب: {e}")
            return pd.DataFrame()


def fetch_native_history(interval, years, symbol=SYMBOL, max_requests=40):
    """
    يسحب تاريخ كافٍ *مباشرة* بالـ interval الأصلي (مش عن طريق resample
    من 5min) — ده اللي بيضمن إن الفريمات الكبيرة زي 1d و 1W تاخد
    فعلاً 10-15 سنة بيانات، مش بس الفترة القصيرة اللي بيغطيها الـ 5min.
    """
    fetcher = TwelveDataFetcher(TWELVE_DATA_API_KEY)
    cutoff = datetime.utcnow() - timedelta(days=int(365 * years))
    end_date = datetime.utcnow()
    all_data, previous_oldest, req = [], None, 0

    while req < max_requests:
        req += 1
        df_batch = fetcher.fetch_range(
            symbol=symbol, interval=interval,
            end_date=end_date.strftime("%Y-%m-%d %H:%M:%S"),
            outputsize=5000,
        )
        if df_batch.empty:
            break
        all_data.append(df_batch)
        oldest = df_batch["datetime"].min()
        logging.info(f"[{interval}] جلب دفعة حتى {oldest} — إجمالي دفعات: {len(all_data)}")
        if previous_oldest == oldest or oldest <= cutoff:
            break
        previous_oldest = oldest
        end_date = oldest - timedelta(seconds=1)
        time.sleep(0.6)

    if not all_data:
        return pd.DataFrame()
    df = pd.concat(all_data, ignore_index=True)
    df.drop_duplicates(subset="datetime", keep="first", inplace=True)
    df = df[df["datetime"] >= cutoff].sort_values("datetime").reset_index(drop=True)

    if not df.empty:
        actual_years = (df["datetime"].max() - df["datetime"].min()).days / 365.0
        logging.info(f"[{interval}] إجمالي الشموع بعد القطع لـ {years} سنة مطلوبة: {len(df)} "
                      f"(تغطية فعلية: {actual_years:.2f} سنة)")
        if actual_years < years * 0.7:
            logging.warning(
                f"[{interval}] ⚠️ التغطية الفعلية ({actual_years:.2f} سنة) أقل بكتير من "
                f"المطلوب ({years} سنة) — الخطة الحالية في TwelveData على الأرجح محدودة "
                f"لهذا الـ interval. إما قلّل قيمة 'years' في TIMEFRAME_CONFIG لتطابق الواقع، "
                f"أو رقّي خطة TwelveData. تدريب على تاريخ قصير بيضعّف جودة الإشارة ويقلل "
                f"عدد الصفقات في كل فولد."
            )
    return df


def get_or_fetch_timeframe_data(tf_key, config, force_refresh=False):
    os.makedirs(DATA_DIR, exist_ok=True)
    csv_path = os.path.join(DATA_DIR, f"{SYMBOL_SAFE}_{tf_key}_raw.csv")
    if os.path.exists(csv_path) and not force_refresh:
        df_old = pd.read_csv(csv_path)
        df_old["datetime"] = pd.to_datetime(df_old["datetime"])
        fetcher = TwelveDataFetcher(TWELVE_DATA_API_KEY)
        last_date = df_old["datetime"].max()
        df_new = fetcher.fetch_range(
            symbol=SYMBOL, interval=config["interval"],
            start_date=(last_date + timedelta(minutes=1)).strftime("%Y-%m-%d %H:%M:%S"),
            end_date=datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S"),
            outputsize=2000,
        )
        if not df_new.empty:
            df_old = pd.concat([df_old, df_new], ignore_index=True)
            df_old.drop_duplicates(subset="datetime", keep="last", inplace=True)
            df_old.sort_values("datetime", inplace=True)
            df_old.reset_index(drop=True, inplace=True)
            df_old.to_csv(csv_path, index=False)
        return df_old

    force_source = config.get("force_resample_source")
    if force_source:
        logging.info(f"[{tf_key}] مضبوط على resample إجباري من {force_source} "
                     f"(تخطي محاولة الجلب المباشر تماماً — كانت بترجع 404/400 من الـ API)")
        df = pd.DataFrame()
    else:
        df = fetch_native_history(config["interval"], config["years"])

    # ─── Fallback: لو الجلب المباشر فشل تماماً (حد تاريخي حقيقي في خطة
    # TwelveData لهذا الـ interval)، نعمل resample من أصغر فريم متاح
    # عندنا بيانات له، بدل ما الفريم ده يفضل من غير أي بيانات إطلاقاً.
    # ده حل بديل مؤقت — التغطية التاريخية هتكون بحدود الفريم الأصغر
    # نفسه، مش أفضل من الجلب المباشر، لكنه أحسن من صفر بيانات.
    if df.empty:
        pandas_freq_map = {
            "5min": "5min", "15min": "15min", "45min": "45min",
            "1h": "1h", "2h": "2h", "4h": "4h", "1day": "1D", "1week": "1W",
        }
        rule = pandas_freq_map.get(config["interval"])
        # نفضّل أقرب فريم أصغر ناجح (تجميع أنظف، ضجيج أقل) قبل ما نلجأ
        # لـ 15min/5min كملاذ أخير. الترتيب هنا مبني على القرب الطبيعي:
        # 4h نفضل نجمعه من 1h مش من 15min مباشرة، وهكذا.
        fallback_chain = {
            "2h": ["1h", "45min", "15min", "5min"],
            "4h": ["1h", "45min", "15min", "5min"],
            "1day": ["4h", "1h", "45min", "15min"],
            "1week": ["1d", "4h", "1h"],
        }
        smaller_candidates = fallback_chain.get(config["interval"], ["15min", "5min"])
        if force_source:
            smaller_candidates = [force_source] + [c for c in smaller_candidates if c != force_source]
        for smaller_tf in smaller_candidates:
            if smaller_tf == tf_key:
                continue
            smaller_csv = os.path.join(DATA_DIR, f"{SYMBOL_SAFE}_{smaller_tf}_raw.csv")
            if os.path.exists(smaller_csv):
                df_small = pd.read_csv(smaller_csv)
                df_small["datetime"] = pd.to_datetime(df_small["datetime"])
            elif smaller_tf in TIMEFRAME_CONFIG:
                df_small = fetch_native_history(
                    TIMEFRAME_CONFIG[smaller_tf]["interval"],
                    TIMEFRAME_CONFIG[smaller_tf]["years"],
                )
            else:
                continue
            if not df_small.empty and rule:
                logging.warning(
                    f"[{tf_key}] الجلب المباشر رجع فاضي (حد تاريخي حقيقي في الخطة الحالية) "
                    f"— هعمل resample من {smaller_tf} كحل بديل. التغطية التاريخية هتكون "
                    f"بحدود {smaller_tf} نفسه، مش أفضل."
                )
                df = (
                    df_small.set_index("datetime")
                    .resample(rule)
                    .agg({"open": "first", "high": "max", "low": "min",
                          "close": "last", "volume": "sum"})
                    .dropna()
                    .reset_index()
                )
                break

    if not df.empty:
        df.to_csv(csv_path, index=False)
    else:
        logging.error(f"[{tf_key}] فشل الحصول على أي بيانات (لا مباشر ولا resample fallback)")
    return df


# ══════════════════════════════════════════════════════════════
# SMC FEATURE ENGINEER
# ══════════════════════════════════════════════════════════════
class SMCFeatureEngineer:
    def __init__(self):
        self.scaler = RobustScaler()
        self.feature_columns = None
        self.scaler_fitted = False

    # ── 1) Time / Sessions / Kill Zones (بتوقيت UTC — الفريمات
    #        كلها جايه من مصدر واحد فالتوقيت النسبي متسق) ──────
    def add_sessions_and_killzones(self, df):
        df = df.copy()
        dt = df["datetime"]
        h = dt.dt.hour
        df["hour"] = h
        df["day_of_week"] = dt.dt.dayofweek
        df["hour_sin"] = np.sin(2 * np.pi * h / 24)
        df["hour_cos"] = np.cos(2 * np.pi * h / 24)

        # GBP/USD sessions — لندن مهيمنة تماماً (الجنيه عملة محلية)
        # سيدني/طوكيو ميتة تقريباً على Cable
        df["is_asia"]         = ((h >= 0) & (h < 7)).astype(int)
        df["is_london"]       = ((h >= 7) & (h < 16)).astype(int)
        df["is_ny"]           = ((h >= 12) & (h < 21)).astype(int)
        df["is_overlap"]      = ((h >= 12) & (h < 16)).astype(int)
        # ─── فيتشورز خاصة بـ GBP/USD ───
        df["is_london_open"]  = ((h == 7)).astype(int)      # ساعة whipsaw شهيرة
        df["is_uk_data_window"] = ((h >= 9) & (h < 10)).astype(int)  # UK CPI/GDP
        df["is_boe_window"]   = ((h >= 11) & (h < 12)).astype(int)  # BoE قرارات
        df["is_london_fix"]   = ((h == 15) & (dt.dt.minute >= 55)).astype(int)  # London Fix 16:00
        df["is_london_ny_prime"] = ((h >= 12) & (h < 16)).astype(int)
        # علامة عامة لأخطر ساعات whipsaws على Cable
        df["is_cable_risky"]  = (((h == 7)) | ((h >= 9) & (h < 12)) | ((h == 15))).astype(int)

        # ICT Kill Zones التقريبية (UTC) — لو عايز دقة بالثانية اربطها
        # بتوقيت نيويورك الفعلي مع مراعاة DST.
        df["kz_london_open"] = ((h >= 7) & (h < 10)).astype(int)
        df["kz_ny_am"]       = ((h >= 12) & (h < 15)).astype(int)
        df["kz_london_close"]= ((h >= 14) & (h < 16)).astype(int)
        df["kz_silver_bullet_am"] = ((h == 14)).astype(int)   # 10-11 NY ≈ 14 UTC
        df["kz_silver_bullet_pm"] = ((h == 19)).astype(int)   # 2-3 NY PM ≈ 19 UTC
        df["is_friday"] = (df["day_of_week"] == 4).astype(int)

        # Asian session range (لسبب Judas Swing عند لندن) — لكل يوم
        df["date_only"] = dt.dt.date
        asia_mask = df["is_asia"] == 1
        asia_high = df.loc[asia_mask].groupby("date_only")["high"].transform("max")
        asia_low  = df.loc[asia_mask].groupby("date_only")["low"].transform("min")
        df["asia_high"] = np.nan
        df["asia_low"] = np.nan
        df.loc[asia_mask, "asia_high"] = asia_high
        df.loc[asia_mask, "asia_low"] = asia_low
        df["asia_high"] = df.groupby("date_only")["asia_high"].transform(
            lambda s: s.ffill().bfill()
        )
        df["asia_low"] = df.groupby("date_only")["asia_low"].transform(
            lambda s: s.ffill().bfill()
        )
        df["dist_to_asia_high"] = (df["asia_high"] - df["close"]) / (df["close"] + 1e-9)
        df["dist_to_asia_low"]  = (df["close"] - df["asia_low"]) / (df["close"] + 1e-9)
        df.drop(columns=["date_only"], inplace=True)
        return df

    # ── 2) Base indicators لازمة للـ confluence (ATR أساسي لكل حاجة) ──
    def compute_base_indicators(self, df):
        df = df.copy()
        c, h, l, o = df["close"], df["high"], df["low"], df["open"]

        tr = pd.concat([
            h - l, (h - c.shift()).abs(), (l - c.shift()).abs()
        ], axis=1).max(axis=1)
        df["atr"] = tr.rolling(14).mean()
        df["atr_pct"] = df["atr"] / c

        df["ema_50"] = c.ewm(span=50).mean()
        df["ema_200"] = c.ewm(span=200).mean()
        df["trend_filter"] = (df["ema_50"] > df["ema_200"]).astype(int)

        df["body_size"] = (c - o).abs()
        df["candle_range"] = (h - l)
        df["upper_wick"] = h - c.where(c >= o, o)
        df["lower_wick"] = c.where(c <= o, o) - l
        df["close_position"] = (c - l) / (h - l + 1e-9)
        df["returns"] = c.pct_change()
        return df

    # ── 3) Swing Points (fractals) — نقاط تأكدت فعلاً (لا lookahead) ──
    def add_swing_points(self, df, lookback=5):
        df = df.copy()
        h, l = df["high"], df["low"]
        n = len(df)
        is_sw_high = np.zeros(n, dtype=int)
        is_sw_low  = np.zeros(n, dtype=int)

        for i in range(lookback, n - lookback):
            window_h = h.iloc[i - lookback:i + lookback + 1]
            window_l = l.iloc[i - lookback:i + lookback + 1]
            if h.iloc[i] == window_h.max() and (window_h == h.iloc[i]).sum() == 1:
                is_sw_high[i] = 1
            if l.iloc[i] == window_l.min() and (window_l == l.iloc[i]).sum() == 1:
                is_sw_low[i] = 1

        # التأكيد بيحصل فعلياً بعد `lookback` شمعة إضافية (لازم نشوف يمين
        # النقطة عشان نتأكد إنها قمة/قاع) — نزيح العلم لقدام بمقدار lookback
        # عشان الفيتشور يبقى معروف "وقت" التأكيد الحقيقي، مش وقت حدوثه.
        confirmed_high = pd.Series(is_sw_high, index=df.index).shift(lookback).fillna(0).astype(int)
        confirmed_low  = pd.Series(is_sw_low,  index=df.index).shift(lookback).fillna(0).astype(int)

        df["swing_high_flag"] = confirmed_high
        df["swing_low_flag"] = confirmed_low
        # قيمة آخر سوينج مؤكد (ffill) — يمثل آخر قمة/قاع "معروفة" في وقتها
        df["last_swing_high"] = df["high"].where(confirmed_high == 1).ffill()
        df["last_swing_low"]  = df["low"].where(confirmed_low == 1).ffill()
        return df

    # ── 4) Market Structure: BOS / CHoCH على أساس السوينجات المؤكدة ──
    def add_market_structure_ict(self, df):
        df = df.copy()
        c = df["close"]
        prev_sw_high = df["last_swing_high"].shift(1)
        prev_sw_low  = df["last_swing_low"].shift(1)

        df["bos_up"]   = (c > prev_sw_high).astype(int)
        df["bos_down"] = (c < prev_sw_low).astype(int)

        # تتبع الاتجاه الحالي: 1 صاعد / -1 هابط / 0 غير محدد
        structure = np.zeros(len(df), dtype=int)
        cur = 0
        bos_up = df["bos_up"].values
        bos_down = df["bos_down"].values
        for i in range(len(df)):
            if bos_up[i] == 1:
                cur = 1
            elif bos_down[i] == 1:
                cur = -1
            structure[i] = cur
        df["market_structure_trend"] = structure

        # CHoCH = كسر عكس اتجاه الترند الحالي (تغيير الشخصية)
        prev_trend = pd.Series(structure, index=df.index).shift(1).fillna(0)
        df["choch_bullish"] = ((df["bos_up"] == 1) & (prev_trend == -1)).astype(int)
        df["choch_bearish"] = ((df["bos_down"] == 1) & (prev_trend == 1)).astype(int)

        df["dist_to_last_swing_high"] = (df["last_swing_high"] - c) / (c + 1e-9)
        df["dist_to_last_swing_low"]  = (c - df["last_swing_low"]) / (c + 1e-9)
        return df

    # ── 5) Fair Value Gaps (3-candle imbalance) ──
    def add_fair_value_gaps(self, df):
        df = df.copy()
        h, l, c, o = df["high"].values, df["low"].values, df["close"].values, df["open"].values
        atr = df["atr"].values
        n = len(df)
        fvg_bull = np.zeros(n); fvg_bear = np.zeros(n)
        fvg_bull_top = np.full(n, np.nan); fvg_bull_bot = np.full(n, np.nan)
        fvg_bear_top = np.full(n, np.nan); fvg_bear_bot = np.full(n, np.nan)
        fvg_bull_quality_at_formation = np.full(n, np.nan)
        fvg_bear_quality_at_formation = np.full(n, np.nan)

        for i in range(2, n):
            a = atr[i] if not np.isnan(atr[i]) and atr[i] > 0 else np.nan
            if l[i] > h[i - 2]:                      # Bullish FVG
                fvg_bull[i] = 1
                fvg_bull_top[i] = l[i]
                fvg_bull_bot[i] = h[i - 2]
                if not np.isnan(a):
                    gap_size_atr = (l[i] - h[i - 2]) / a
                    # الشمعة اللي كوّنت الفجوة (i-1 عادة) كانت إزاحة حقيقية
                    # (جسم كبير) ولا حركة عادية؟ فجوة من إزاحة أقوى بكتير.
                    origin_body = abs(c[i - 1] - o[i - 1])
                    is_disp_origin = 1.0 if origin_body > 1.5 * a else 0.5
                    fvg_bull_quality_at_formation[i] = min(gap_size_atr, 2.0) / 2.0 * is_disp_origin
            if h[i] < l[i - 2]:                       # Bearish FVG
                fvg_bear[i] = 1
                fvg_bear_top[i] = l[i - 2]
                fvg_bear_bot[i] = h[i]
                if not np.isnan(a):
                    gap_size_atr = (l[i - 2] - h[i]) / a
                    origin_body = abs(c[i - 1] - o[i - 1])
                    is_disp_origin = 1.0 if origin_body > 1.5 * a else 0.5
                    fvg_bear_quality_at_formation[i] = min(gap_size_atr, 2.0) / 2.0 * is_disp_origin

        df["fvg_bullish"] = fvg_bull.astype(int)
        df["fvg_bearish"] = fvg_bear.astype(int)

        # آخر FVG معروف (ffill) + نسبة امتلائه بالسعر الحالي
        last_bull_top = pd.Series(fvg_bull_top, index=df.index).ffill()
        last_bull_bot = pd.Series(fvg_bull_bot, index=df.index).ffill()
        last_bear_top = pd.Series(fvg_bear_top, index=df.index).ffill()
        last_bear_bot = pd.Series(fvg_bear_bot, index=df.index).ffill()
        last_bull_quality = pd.Series(fvg_bull_quality_at_formation, index=df.index).ffill().fillna(0)
        last_bear_quality = pd.Series(fvg_bear_quality_at_formation, index=df.index).ffill().fillna(0)

        c_s = df["close"]
        bull_range = (last_bull_top - last_bull_bot).replace(0, np.nan)
        bear_range = (last_bear_top - last_bear_bot).replace(0, np.nan)
        df["dist_to_bull_fvg"] = (c_s - last_bull_bot) / (c_s + 1e-9)
        df["dist_to_bear_fvg"] = (last_bear_top - c_s) / (c_s + 1e-9)
        df["bull_fvg_fill_pct"] = ((last_bull_top - c_s) / bull_range).clip(0, 1).fillna(0)
        df["bear_fvg_fill_pct"] = ((c_s - last_bear_bot) / bear_range).clip(0, 1).fillna(0)

        # FVG Quality Score: يدمج حجم الفجوة نسبة لـ ATR (فجوة كبيرة =
        # اختلال أوضح)، حداثة الامتلاء (فجوة اتلمت 90% ضعيفة)، ومصدر
        # الفجوة (اتكونت من شمعة إزاحة قوية ولا حركة عادية).
        df["fvg_bull_quality"] = (last_bull_quality * (1 - df["bull_fvg_fill_pct"])).clip(0, 1)
        df["fvg_bear_quality"] = (last_bear_quality * (1 - df["bear_fvg_fill_pct"])).clip(0, 1)
        return df

    # ── 6) Order Blocks (آخر شمعة معاكسة قبل حركة إزاحة تكسر الهيكل) ──
    def add_order_blocks(self, df, disp_mult=1.5, max_lookback=6):
        """
        نسخة محسّنة عن الأول: بدل ما ناخد الشمعة اللي قبل الإزاحة مباشرة
        (اللي ممكن تكون مش الـ OB الحقيقي لو فيه أكتر من شمعة بنفس
        الاتجاه قبل الانعكاس)، بندور فعلياً على *آخر* شمعة معاكسة قبل
        شمعة الإزاحة (زي تعريف ICT الأصلي)، وبنضيف:
          - ob_strength: قوة الـ OB نسبة لحجم شمعة الإزاحة اللي كوّنته
          - mitigated: هل السعر رجع لمنطقة الـ OB بعد تكوينه
          - breaker: OB اتـ mitigated وبعدين السعر قفل خارجها بالكامل
            (يعني اتكسر فعلاً وممكن يتحول لدعم/مقاومة معاكسة)
        """
        df = df.copy()
        o, c, h, l = df["open"].values, df["close"].values, df["high"].values, df["low"].values
        atr = df["atr"].values
        bos_up = df["bos_up"].values
        bos_down = df["bos_down"].values
        n = len(df)

        bull_ob_top = np.full(n, np.nan); bull_ob_bot = np.full(n, np.nan)
        bear_ob_top = np.full(n, np.nan); bear_ob_bot = np.full(n, np.nan)
        bull_ob_strength = np.full(n, np.nan); bear_ob_strength = np.full(n, np.nan)
        bull_mitigated = np.zeros(n, dtype=int); bear_mitigated = np.zeros(n, dtype=int)
        bull_breaker = np.zeros(n, dtype=int); bear_breaker = np.zeros(n, dtype=int)
        bull_ob_age = np.full(n, np.nan); bear_ob_age = np.full(n, np.nan)
        bull_breaker_strength = np.full(n, np.nan); bear_breaker_strength = np.full(n, np.nan)

        cur_bull_top = cur_bull_bot = np.nan
        cur_bear_top = cur_bear_bot = np.nan
        cur_bull_mitigated = cur_bear_mitigated = False
        cur_bull_broken = cur_bear_broken = False
        cur_bull_formed_at = cur_bear_formed_at = None
        cur_bull_breaker_strength = cur_bear_breaker_strength = np.nan

        for i in range(1, n):
            body = abs(c[i] - o[i])
            is_displacement = (not np.isnan(atr[i])) and atr[i] > 0 and body > disp_mult * atr[i]

            # Bullish OB: امسح لورا لحد آخر شمعة هابطة حقيقية قبل الإزاحة
            if bos_up[i] == 1 and is_displacement and c[i] > o[i]:
                for back in range(1, max_lookback + 1):
                    j = i - back
                    if j < 0:
                        break
                    if c[j] < o[j]:  # أول شمعة هابطة نلاقيها بالرجوع للخلف
                        cur_bull_top, cur_bull_bot = h[j], l[j]
                        cur_bull_mitigated, cur_bull_broken = False, False
                        cur_bull_formed_at = i
                        cur_bull_breaker_strength = np.nan
                        break
                    if c[j] > o[j] and back > 1:
                        # سلسلة شموع صاعدة متتالية — نوقف الدوران، مفيش OB واضح
                        break

            # Bearish OB: نفس المنطق بس معكوس
            if bos_down[i] == 1 and is_displacement and c[i] < o[i]:
                for back in range(1, max_lookback + 1):
                    j = i - back
                    if j < 0:
                        break
                    if c[j] > o[j]:
                        cur_bear_top, cur_bear_bot = h[j], l[j]
                        cur_bear_mitigated, cur_bear_broken = False, False
                        cur_bear_formed_at = i
                        cur_bear_breaker_strength = np.nan
                        break
                    if c[j] < o[j] and back > 1:
                        break

            # تتبع الـ mitigation: هل السعر الحالي رجع لمنطقة الـ OB بعد تكوينه؟
            if not np.isnan(cur_bull_bot) and l[i] <= cur_bull_top and not cur_bull_mitigated:
                cur_bull_mitigated = True
            if not np.isnan(cur_bear_top) and h[i] >= cur_bear_bot and not cur_bear_mitigated:
                cur_bear_mitigated = True

            # Breaker: اتـ mitigated وبعدين السعر قفل بره المنطقة بالكامل (اتكسر فعلاً)
            # القوة = مسافة الإغلاق وراء حدود الـ OB نسبة لـ ATR — كسر
            # بمسافة كبيرة (زخم قوي) أقوى بكتير من كسر بالكاد عدّى الحد.
            if cur_bull_mitigated and not np.isnan(cur_bull_bot) and c[i] < cur_bull_bot:
                cur_bull_broken = True
                if not np.isnan(atr[i]) and atr[i] > 0:
                    cur_bull_breaker_strength = min((cur_bull_bot - c[i]) / atr[i], 3.0) / 3.0
            if cur_bear_mitigated and not np.isnan(cur_bear_top) and c[i] > cur_bear_top:
                cur_bear_broken = True
                if not np.isnan(atr[i]) and atr[i] > 0:
                    cur_bear_breaker_strength = min((c[i] - cur_bear_top) / atr[i], 3.0) / 3.0

            bull_ob_top[i], bull_ob_bot[i] = cur_bull_top, cur_bull_bot
            bear_ob_top[i], bear_ob_bot[i] = cur_bear_top, cur_bear_bot
            if not np.isnan(atr[i]) and atr[i] > 0:
                if not np.isnan(cur_bull_top):
                    bull_ob_strength[i] = abs(c[i - 1] - o[i - 1]) / atr[i] if i > 0 else np.nan
                if not np.isnan(cur_bear_top):
                    bear_ob_strength[i] = abs(c[i - 1] - o[i - 1]) / atr[i] if i > 0 else np.nan
            bull_mitigated[i] = int(cur_bull_mitigated)
            bear_mitigated[i] = int(cur_bear_mitigated)
            bull_breaker[i] = int(cur_bull_broken)
            bear_breaker[i] = int(cur_bear_broken)
            bull_ob_age[i] = (i - cur_bull_formed_at) if cur_bull_formed_at is not None else np.nan
            bear_ob_age[i] = (i - cur_bear_formed_at) if cur_bear_formed_at is not None else np.nan
            bull_breaker_strength[i] = cur_bull_breaker_strength
            bear_breaker_strength[i] = cur_bear_breaker_strength

        c_s = df["close"]
        df["bull_ob_top"] = bull_ob_top
        df["bull_ob_bot"] = bull_ob_bot
        df["bear_ob_top"] = bear_ob_top
        df["bear_ob_bot"] = bear_ob_bot
        df["bull_ob_strength"] = bull_ob_strength
        df["bear_ob_strength"] = bear_ob_strength
        df["bull_ob_mitigated"] = bull_mitigated
        df["bear_ob_mitigated"] = bear_mitigated
        df["bull_breaker_block"] = bull_breaker
        df["bear_breaker_block"] = bear_breaker
        df["dist_to_bull_ob"] = (c_s - df["bull_ob_bot"]) / (c_s + 1e-9)
        df["dist_to_bear_ob"] = (df["bear_ob_top"] - c_s) / (c_s + 1e-9)
        df["price_inside_bull_ob"] = (
            (c_s <= df["bull_ob_top"]) & (c_s >= df["bull_ob_bot"]) & (bull_mitigated == 0)
        ).astype(int)
        df["price_inside_bear_ob"] = (
            (c_s <= df["bear_ob_top"]) & (c_s >= df["bear_ob_bot"]) & (bear_mitigated == 0)
        ).astype(int)

        # Mitigation Block: OB اتلمس قبل كده (mitigated) — منطقة رجعة
        # شائعة قبل الاستمرار، مختلفة عن OB الطازج. بنعلّم لو السعر حالياً
        # راجع داخل منطقة OB اتلمست قبل كده (إعادة اختبار).
        df["in_mitigation_zone_bull"] = (
            (c_s <= df["bull_ob_top"]) & (c_s >= df["bull_ob_bot"]) & (bull_mitigated == 1)
        ).astype(int)
        df["in_mitigation_zone_bear"] = (
            (c_s <= df["bear_ob_top"]) & (c_s >= df["bear_ob_bot"]) & (bear_mitigated == 1)
        ).astype(int)

        # Breaker Block: مسافة السعر عن أقرب Breaker — لو قريب منها،
        # ده منطقة يُتوقع تتحول لدعم/مقاومة معاكسة للـ OB الأصلي.
        bull_breaker_level = pd.Series(
            np.where(bull_breaker == 1, bull_ob_bot, np.nan), index=df.index
        ).ffill()
        bear_breaker_level = pd.Series(
            np.where(bear_breaker == 1, bear_ob_top, np.nan), index=df.index
        ).ffill()
        df["dist_to_bull_breaker"] = ((c_s - bull_breaker_level) / (c_s + 1e-9)).fillna(0)
        df["dist_to_bear_breaker"] = ((bear_breaker_level - c_s) / (c_s + 1e-9)).fillna(0)

        # Order Block Age/Freshness: OB لسه عمره قليل (شمعات قليلة من
        # تكوينه) أقوى بكتير من OB عدى عليه وقت طويل من غير ما يتلمس أو
        # يتكسر — الفريشنس بتتناقص أسّياً مع الوقت (half-life ~20 شمعة).
        df["bull_ob_age"] = pd.Series(bull_ob_age, index=df.index).fillna(999)
        df["bear_ob_age"] = pd.Series(bear_ob_age, index=df.index).fillna(999)
        df["bull_ob_freshness"] = np.exp(-df["bull_ob_age"] / 20.0)
        df["bear_ob_freshness"] = np.exp(-df["bear_ob_age"] / 20.0)

        # Breaker Block Strength: مسافة الكسر الفعلي وراء حدود OB نسبة
        # لـ ATR — بيفرّق بين breaker قوي (زخم واضح خلف الكسر) وbreaker
        # ضعيف (بالكاد عدّى الحد ممكن يرجع تاني بسهولة).
        df["bull_breaker_strength"] = pd.Series(bull_breaker_strength, index=df.index).fillna(0)
        df["bear_breaker_strength"] = pd.Series(bear_breaker_strength, index=df.index).fillna(0)
        return df

    # ── 7) Liquidity: Equal Highs/Lows + Sweeps (صيد الوقف) ──
    def add_liquidity(self, df, eq_tol=0.0005, sweep_lookback=20):
        df = df.copy()
        h, l, c = df["high"], df["low"], df["close"]

        roll_high = h.rolling(sweep_lookback).max()
        roll_low = l.rolling(sweep_lookback).min()

        # Equal highs/lows: القمة الحالية قريبة جداً من أعلى قمة بالنافذة
        df["eq_highs"] = ((roll_high - h).abs() / (c + 1e-9) < eq_tol).astype(int)
        df["eq_lows"]  = ((roll_low - l).abs() / (c + 1e-9) < eq_tol).astype(int)

        prev_roll_high = roll_high.shift(1)
        prev_roll_low = roll_low.shift(1)
        # Sweep up: كسر أعلى القمم بالفتيل ثم إغلاق تحتها (صيد سيولة شراء)
        df["liquidity_sweep_up"] = (
            (h > prev_roll_high) & (c < prev_roll_high)
        ).astype(int)
        # Sweep down: كسر أدنى القيعان بالفتيل ثم إغلاق فوقها
        df["liquidity_sweep_down"] = (
            (l < prev_roll_low) & (c > prev_roll_low)
        ).astype(int)

        df["dist_to_buyside_liquidity"] = (roll_high - c) / (c + 1e-9)
        df["dist_to_sellside_liquidity"] = (c - roll_low) / (c + 1e-9)
        return df

    # ── 8) Premium / Discount + OTE (61.8%-79% فيبوناتشي) ──
    def add_premium_discount(self, df):
        df = df.copy()
        c = df["close"]
        swing_high = df["last_swing_high"]
        swing_low = df["last_swing_low"]
        rng = (swing_high - swing_low).replace(0, np.nan)

        pos = (c - swing_low) / rng
        df["pd_position"] = pos.clip(0, 1).fillna(0.5)
        df["is_premium"] = (df["pd_position"] > 0.5).astype(int)
        df["is_discount"] = (df["pd_position"] <= 0.5).astype(int)
        # OTE: منطقة الدخول المثلى — خصم عميق (شراء) أو خصم علوي (بيع)
        df["in_ote_bullish"] = ((df["pd_position"] >= 0.21) & (df["pd_position"] <= 0.38)).astype(int)
        df["in_ote_bearish"] = ((df["pd_position"] >= 0.62) & (df["pd_position"] <= 0.79)).astype(int)
        return df

    # ── 9) Displacement candles (حركة إزاحة مؤسساتية) ──
    def add_displacement(self, df, mult=1.5):
        df = df.copy()
        body = df["body_size"]
        df["displacement"] = (body > mult * df["atr"]).astype(int)
        df["displacement_bullish"] = (
            (df["displacement"] == 1) & (df["close"] > df["open"])
        ).astype(int)
        df["displacement_bearish"] = (
            (df["displacement"] == 1) & (df["close"] < df["open"])
        ).astype(int)

        # Displacement Confirmation: شمعة الإزاحة لوحدها ممكن تكون فخ —
        # التأكيد الحقيقي إن الشمعة اللي بعدها تكمل نفس الاتجاه (مش ترجع
        # فوراً). ده بيفرّق بين إزاحة مؤسساتية حقيقية وحركة عابرة.
        next_close = df["close"].shift(-1)
        next_open = df["open"].shift(-1)
        df["displacement_confirmed_bull"] = (
            (df["displacement_bullish"] == 1) & (next_close > next_open) &
            (next_close > df["close"])
        ).astype(int)
        df["displacement_confirmed_bear"] = (
            (df["displacement_bearish"] == 1) & (next_close < next_open) &
            (next_close < df["close"])
        ).astype(int)
        return df

    # ── 10) Inducement / Fakeout: كسر وهمي لمستوى ثانوي (سوينج صغير)
    #         بيجذب المتداولين قبل الحركة الحقيقية، ثم السعر بيرتد فوراً
    #         من غير استمرار — ده مختلف عن الـ BOS الحقيقي على السوينج
    #         الرئيسي. ──
    def add_inducement(self, df, minor_lookback=2, revert_window=3):
        df = df.copy()
        h, l, c = df["high"].values, df["low"].values, df["close"].values
        n = len(df)
        is_minor_high = np.zeros(n, dtype=int)
        is_minor_low = np.zeros(n, dtype=int)
        for i in range(minor_lookback, n - minor_lookback):
            wh = h[i - minor_lookback:i + minor_lookback + 1]
            wl = l[i - minor_lookback:i + minor_lookback + 1]
            if h[i] == wh.max() and (wh == h[i]).sum() == 1:
                is_minor_high[i] = 1
            if l[i] == wl.min() and (wl == l[i]).sum() == 1:
                is_minor_low[i] = 1
        confirmed_minor_high = pd.Series(is_minor_high, index=df.index).shift(minor_lookback).fillna(0).astype(int)
        confirmed_minor_low = pd.Series(is_minor_low, index=df.index).shift(minor_lookback).fillna(0).astype(int)
        last_minor_high = df["high"].where(confirmed_minor_high == 1).ffill()
        last_minor_low = df["low"].where(confirmed_minor_low == 1).ffill()

        inducement_bull = np.zeros(n, dtype=int)  # كسر وهمي لقاع ثانوي ثم ارتداد صاعد
        inducement_bear = np.zeros(n, dtype=int)  # كسر وهمي لقمة ثانوية ثم ارتداد هابط
        lml = last_minor_low.values
        lmh = last_minor_high.values
        for i in range(revert_window, n):
            # هل حصل كسر لقاع ثانوي في الشمعات الأخيرة، وبعدها السعر
            # رجع فوق المستوى بسرعة (خلال revert_window) من غير استمرار هبوط؟
            if not np.isnan(lml[i - revert_window]) and l[i - revert_window:i].min() < lml[i - revert_window] and c[i] > lml[i - revert_window]:
                inducement_bull[i] = 1
            if not np.isnan(lmh[i - revert_window]) and h[i - revert_window:i].max() > lmh[i - revert_window] and c[i] < lmh[i - revert_window]:
                inducement_bear[i] = 1

        df["inducement_bull"] = inducement_bull
        df["inducement_bear"] = inducement_bear
        return df

    # ── 11) Liquidity Void / Raid: فراغ سعري (منطقة اتقطعت بسرعة من غير
    #          تداول كافي فيها — شبيهة بالـ FVG لكن أوسع) + Raid (اختراق
    #          عنيف لمنطقة سيولة، أقوى من مجرد sweep عادي). ──
    def add_liquidity_void_raid(self, df, void_atr_mult=2.0, raid_atr_mult=0.5):
        df = df.copy()
        atr_safe = df["atr"].replace(0, np.nan)
        # Void: شمعة واحدة بحركة ضخمة (أكبر من FVG العادي) بتسيب فراغ
        # سعري خلفها لسه ما اتلمسش.
        df["liquidity_void"] = (df["candle_range"] > void_atr_mult * df["atr"]).astype(int)
        void_top = df["high"].where(df["liquidity_void"] == 1).ffill()
        void_bot = df["low"].where(df["liquidity_void"] == 1).ffill()
        df["dist_to_liquidity_void"] = ((df["close"] - void_bot) / (df["close"] + 1e-9)).fillna(0)
        # Void Strength: مش بس "فيه فراغ ولا لأ" — حجم الفراغ نفسه نسبة
        # لـ ATR، فراغ 4×ATR أقوى بكتير من فراغ 2.1×ATR بالكاد.
        df["liquidity_void_strength"] = ((df["candle_range"] / atr_safe).fillna(0) / void_atr_mult).clip(0, 3)

        # Raid: sweep عنيف (الفتيل تعدى مستوى السيولة بمسافة معتبرة نسبة
        # لـ ATR، مش مجرد تعدي بسيط) — يميّز صيد سيولة قوي عن اختراق ضعيف.
        upper_wick_atr = (df.get("upper_wick", pd.Series(0, index=df.index)) / atr_safe).fillna(0)
        lower_wick_atr = (df.get("lower_wick", pd.Series(0, index=df.index)) / atr_safe).fillna(0)
        df["liquidity_raid_up"] = (
            (df.get("liquidity_sweep_up", 0) == 1) & (upper_wick_atr > raid_atr_mult)
        ).astype(int)
        df["liquidity_raid_down"] = (
            (df.get("liquidity_sweep_down", 0) == 1) & (lower_wick_atr > raid_atr_mult)
        ).astype(int)
        # Raid Strength: قوة الاختراق الفعلية (مش بس تحقق شرط أدنى) —
        # فتيل 2×ATR أقوى بكتير من فتيل 0.51×ATR بالكاد عدّى العتبة.
        df["liquidity_raid_up_strength"] = (df["liquidity_raid_up"] * upper_wick_atr).clip(0, 3)
        df["liquidity_raid_down_strength"] = (df["liquidity_raid_down"] * lower_wick_atr).clip(0, 3)

        # Liquidity Grab Confirmation: صيد السيولة (Raid/Sweep) لوحده
        # ممكن يكون ضجيج — التأكيد الحقيقي إن حركة انعكاس فعلية (CHoCH
        # أو إزاحة مؤكدة) بتتبعه خلال شمعات قليلة، مش مجرد فتيل عابر.
        confirm_window = 3
        raid_down_recent = df.get("liquidity_raid_down", pd.Series(0, index=df.index)).rolling(
            confirm_window, min_periods=1).max()
        raid_up_recent = df.get("liquidity_raid_up", pd.Series(0, index=df.index)).rolling(
            confirm_window, min_periods=1).max()
        reversal_signal_bull = (
            (df.get("choch_bullish", 0) == 1) | (df.get("displacement_confirmed_bull", 0) == 1)
        ).astype(int)
        reversal_signal_bear = (
            (df.get("choch_bearish", 0) == 1) | (df.get("displacement_confirmed_bear", 0) == 1)
        ).astype(int)
        df["liquidity_grab_confirmed_bull"] = (
            (raid_down_recent == 1) & (reversal_signal_bull == 1)
        ).astype(int)
        df["liquidity_grab_confirmed_bear"] = (
            (raid_up_recent == 1) & (reversal_signal_bear == 1)
        ).astype(int)
        return df

    # ── 12) Session Momentum: زخم السعر من بداية الجلسة الحالية (آسيا/
    #          لندن/نيويورك) — يفيد في تأكيد اتجاه الجلسة الحالية. ──
    def add_session_momentum(self, df):
        df = df.copy()
        df["_date_only"] = df["datetime"].dt.date
        df["_session_id"] = (
            df["_date_only"].astype(str) + "_" +
            np.where(df["is_asia"] == 1, "asia",
            np.where(df["is_london"] == 1, "london",
            np.where(df["is_ny"] == 1, "ny", "other")))
        )
        session_open = df.groupby("_session_id")["open"].transform("first")
        df["session_momentum"] = ((df["close"] - session_open) / (session_open + 1e-9))
        df["session_momentum_atr"] = (df["close"] - session_open) / (df["atr"].replace(0, np.nan) + 1e-9)
        df["session_momentum_atr"] = df["session_momentum_atr"].fillna(0)
        df.drop(columns=["_date_only", "_session_id"], inplace=True)
        return df

    # ── 10) Confluence Score: دمج كل إشارات SMC في مقياس واحد لكل اتجاه.
    #         بدل ما الموديل يحاول يوازن بين عشرات الأعلام المنفصلة بنفسه،
    #         بنديله مقياس مُجمَّع جاهز بيعبّر عن "قد إيه الإعداد قوي" —
    #         ده بيقلل الحمل على الموديل ويوضحله الإشارة المركّبة. ──
    def add_confluence_score(self, df):
        df = df.copy()
        fresh_bull_ob = (df.get("price_inside_bull_ob", 0) == 1).astype(int)
        fresh_bear_ob = (df.get("price_inside_bear_ob", 0) == 1).astype(int)
        ob_strength_bull = df.get("bull_ob_strength", pd.Series(0, index=df.index)).fillna(0).clip(0, 5) / 5
        ob_strength_bear = df.get("bear_ob_strength", pd.Series(0, index=df.index)).fillna(0).clip(0, 5) / 5

        # FVG بتفقد قيمتها كإشارة كل ما اتلمت أكتر — فجوة اتلمت 80% مش
        # نفس قوة فجوة لسه طازجة 100%. بنستخدم fvg_bull/bear_quality
        # المُحسوبة مسبقاً (بتدمج حجم الفجوة نسبة لـ ATR + الحداثة +
        # مصدر التكوين) بدل مقياس تقريبي محدود.
        fvg_bull_component = df.get("fvg_bull_quality", pd.Series(0, index=df.index)).fillna(0)
        fvg_bear_component = df.get("fvg_bear_quality", pd.Series(0, index=df.index)).fillna(0)

        # اتفاق مع Premium/Discount: شراء في Discount وبيع في Premium بيقوّي
        # الإعداد؛ شراء في Premium (عكس المنطق) بيضعّفه بدل ما يتجاهل.
        pd_pos = df.get("pd_position", pd.Series(0.5, index=df.index))
        pd_align_bull = (1 - pd_pos).clip(0, 1)   # كل ما كنا في خصم أعمق كل ما أحسن للشراء
        pd_align_bear = pd_pos.clip(0, 1)         # كل ما كنا في علاوة أعلى كل ما أحسن للبيع

        # قوة الـ liquidity sweep نفسها (مش بس حصل ولا لأ) عن طريق حجم
        # الفتيل نسبة للـ ATR — sweep بفتيل كبير أقوى من sweep بفتيل بسيط.
        atr_safe = df.get("atr", pd.Series(np.nan, index=df.index)).replace(0, np.nan)
        upper_wick_atr = (df.get("upper_wick", pd.Series(0, index=df.index)) / atr_safe).fillna(0).clip(0, 3) / 3
        lower_wick_atr = (df.get("lower_wick", pd.Series(0, index=df.index)) / atr_safe).fillna(0).clip(0, 3) / 3
        sweep_down_strength = df.get("liquidity_sweep_down", 0) * (0.5 + 0.5 * lower_wick_atr)
        sweep_up_strength = df.get("liquidity_sweep_up", 0) * (0.5 + 0.5 * upper_wick_atr)

        # Liquidity Grab Probability: مقياس SMC مركزي — احتمالية إن السعر
        # حالياً في منطقة صيد سيولة (قرب EQH/EQL + فتيل قوي + قرب حد
        # أعلى/أدنى نطاق حديث). ده جوهر "نموذج رسم السيولة" في SMC —
        # السعر بيتحرك بحثاً عن سيولة قبل الحركة الحقيقية، فالاحتمال ده
        # بيوضح "هل احنا في منطقة صيد محتملة دلوقتي؟"
        eq_component = (df.get("eq_highs", 0).astype(float) + df.get("eq_lows", 0).astype(float)).clip(0, 1)
        dist_buyside = df.get("dist_to_buyside_liquidity", pd.Series(0, index=df.index)).abs()
        dist_sellside = df.get("dist_to_sellside_liquidity", pd.Series(0, index=df.index)).abs()
        proximity_score = (1 - (dist_buyside.clip(0, 0.01) / 0.01)).clip(0, 1) * 0.5 + \
                           (1 - (dist_sellside.clip(0, 0.01) / 0.01)).clip(0, 1) * 0.5
        df["liquidity_grab_probability"] = (
            0.4 * eq_component + 0.3 * proximity_score +
            0.15 * (upper_wick_atr + lower_wick_atr).clip(0, 1) +
            0.15 * (df.get("liquidity_raid_up", 0) + df.get("liquidity_raid_down", 0)).clip(0, 1)
        ).clip(0, 1)

        # OB الطازج بقى موزون بحداثته (Freshness) — نفس الـ OB لكن عمره
        # 40 شمعة من غير أي لمسة أضعف من نفس الـ OB عمره شمعتين بس.
        ob_freshness_bull = df.get("bull_ob_freshness", pd.Series(1.0, index=df.index)).fillna(1.0)
        ob_freshness_bear = df.get("bear_ob_freshness", pd.Series(1.0, index=df.index)).fillna(1.0)

        # مقياس سيولة مُجمَّع (Liquidity component) — بيدمج قوة الـ sweep،
        # قوة الـ raid، وقوة الـ void كلهم في مكوّن واحد بوزن أعلى، بدل
        # ما يتوزعوا مبعثرين بأوزان صغيرة منفصلة.
        void_strength = df.get("liquidity_void_strength", pd.Series(0, index=df.index)).fillna(0) / 3
        raid_down_s = df.get("liquidity_raid_down_strength", pd.Series(0, index=df.index)).fillna(0) / 3
        raid_up_s = df.get("liquidity_raid_up_strength", pd.Series(0, index=df.index)).fillna(0) / 3
        liquidity_component_bull = (0.5 * sweep_down_strength + 0.3 * raid_down_s + 0.2 * void_strength).clip(0, 1)
        liquidity_component_bear = (0.5 * sweep_up_strength + 0.3 * raid_up_s + 0.2 * void_strength).clip(0, 1)
        # Grab لوحده أضعف من Grab متأكد بحركة انعكاس فعلية بعده — بنضيف
        # bonus بدل ما نعتمد على الـ raid الخام بس.
        liquidity_component_bull = (
            liquidity_component_bull + 0.3 * df.get("liquidity_grab_confirmed_bull", 0)
        ).clip(0, 1)
        liquidity_component_bear = (
            liquidity_component_bear + 0.3 * df.get("liquidity_grab_confirmed_bear", 0)
        ).clip(0, 1)
        breaker_component_bull = df.get("bull_breaker_strength", pd.Series(0, index=df.index)).fillna(0)
        breaker_component_bear = df.get("bear_breaker_strength", pd.Series(0, index=df.index)).fillna(0)
        df["smc_confluence_bull"] = (
            0.12 * fvg_bull_component
            + 0.20 * (fresh_bull_ob * ob_freshness_bull)
            + 0.09 * ob_strength_bull
            + 0.07 * df.get("in_ote_bullish", 0)
            + 0.05 * pd_align_bull
            + 0.09 * (df.get("displacement_confirmed_bull", df.get("displacement_bullish", 0)) * df.get("bos_up", 0))
            + 0.19 * liquidity_component_bull
            + 0.04 * df.get("choch_bullish", 0)
            + 0.05 * df.get("inducement_bull", 0)
            + 0.04 * df.get("in_mitigation_zone_bull", 0)
            + 0.06 * breaker_component_bull
        ).clip(0, 1.5)

        df["smc_confluence_bear"] = (
            0.12 * fvg_bear_component
            + 0.20 * (fresh_bear_ob * ob_freshness_bear)
            + 0.09 * ob_strength_bear
            + 0.07 * df.get("in_ote_bearish", 0)
            + 0.05 * pd_align_bear
            + 0.09 * (df.get("displacement_confirmed_bear", df.get("displacement_bearish", 0)) * df.get("bos_down", 0))
            + 0.19 * liquidity_component_bear
            + 0.04 * df.get("choch_bearish", 0)
            + 0.05 * df.get("inducement_bear", 0)
            + 0.04 * df.get("in_mitigation_zone_bear", 0)
            + 0.06 * breaker_component_bear
        ).clip(0, 1.5)

        df["smc_confluence_net"] = df["smc_confluence_bull"] - df["smc_confluence_bear"]
        # بوابة جودة: هل الإعداد قوي بما يكفي (≥3 عناصر متفقة تقريباً)؟
        # ده فيتشور مباشر يقدر الموديل يستخدمه كفلتر بدل ما يستنتجه بنفسه.
        df["smc_high_quality_bull"] = (df["smc_confluence_bull"] >= 0.6).astype(int)
        df["smc_high_quality_bear"] = (df["smc_confluence_bear"] >= 0.6).astype(int)
        return df

    # ── MAIN PIPELINE ──
    def create_features(self, df, config, shift=True):
        df = self.add_sessions_and_killzones(df)
        df = self.compute_base_indicators(df)
        df = self.add_swing_points(df, lookback=config.get("swing_lookback", 5))
        df = self.add_market_structure_ict(df)
        df = self.add_fair_value_gaps(df)
        df = self.add_order_blocks(df)
        df = self.add_liquidity(df)
        df = self.add_premium_discount(df)
        df = self.add_displacement(df)
        df = self.add_inducement(df)
        df = self.add_liquidity_void_raid(df)
        df = self.add_session_momentum(df)
        df = self.add_confluence_score(df)

        exclude_from_shift = [
            "datetime", "open", "high", "low", "close", "volume",
            "hour", "day_of_week", "hour_sin", "hour_cos",
            "is_asia", "is_london", "is_ny", "is_overlap", "is_friday",
            "kz_london_open", "kz_ny_am", "kz_london_close",
            "kz_silver_bullet_am", "kz_silver_bullet_pm",
        ]
        tech_cols = [c for c in df.columns if c not in exclude_from_shift]
        if shift:
            for col in tech_cols:
                df[col] = df[col].shift(1)

        thresh = int(len(df) * 0.5)
        df = df.dropna(axis=1, thresh=thresh)
        warmup = max(50, len(df) // 20)
        df = df.iloc[warmup:].copy()
        df = df.ffill().fillna(0)

        exclude_final = ["datetime", "open", "high", "low", "close", "volume",
                          "future_close", "pct_change", "target"]
        self.feature_columns = [c for c in df.columns if c not in exclude_final]
        return df

    def create_target(self, df, horizon=1, min_atr_mult=None, threshold=None):
        """
        بدل ما نصنّف كل شمعة كـ up/down حتى لو الحركة تافهة (ضجيج)،
        بنفلتر بـ min_atr_mult: أي حركة أصغر من (min_atr_mult × ATR وقت
        الإشارة) بتتشال خالص من عينة التدريب — مش بتتحسب up ولا down.
        ده بيقلل noise الهدف بشكل كبير ويخلي الموديل يتعلم بس من حركات
        واضحة فعلاً، مش تذبذب عشوائي حوالي الصفر.
        """
        df = df.copy()
        df["future_close"] = df["close"].shift(-horizon)
        df["pct_change"] = (df["future_close"] - df["close"]) / df["close"]
        move = df["future_close"] - df["close"]

        if threshold is not None:
            df["target"] = np.where(df["pct_change"] > threshold, 2,
                            np.where(df["pct_change"] < -threshold, 0, 1))
            df.dropna(subset=["future_close"], inplace=True)
            df["target"] = df["target"].astype(int)
            return df

        if min_atr_mult and "atr" in df.columns:
            move_thresh = min_atr_mult * df["atr"]
            raw_target = np.where(move > move_thresh, 1,
                          np.where(move < -move_thresh, 0, -1))  # -1 = ضجيج، هيتشال
            df["target"] = raw_target
            n_before = len(df)
            df = df[df["target"] != -1].copy()
            n_after = len(df)
            logging.info(f"فلترة الهدف بـ ATR×{min_atr_mult}: اتشال {n_before - n_after} "
                          f"صف ضجيج من أصل {n_before} ({(n_before-n_after)/max(n_before,1)*100:.1f}%)")
        else:
            df["target"] = (df["future_close"] > df["close"]).astype(int)

        df.dropna(subset=["future_close"], inplace=True)
        df["target"] = df["target"].astype(int)
        return df

    def create_target_triple_barrier(self, df, horizon, atr_sl_mult, atr_tp_mult):
        """
        Triple-barrier labeling (زي طريقة López de Prado المعروفة في الكمّي):
        بدل ما نسأل "هل السعر أعلى بعد N شمعة؟" (سؤال ضعيف وضجيج عالي)،
        بنسأل السؤال اللي فعلاً بيحصل وقت التداول: "لو فتحنا صفقة شراء
        دلوقتي بنفس TP/SL اللي الباك تيستر هيستخدمهم، هل TP هيتحقق قبل
        SL خلال أفق الأفق الزمني ده؟"
        - target=1: TP اتحقق الأول (صفقة رابحة)
        - target=0: SL اتحقق الأول (صفقة خسرانة)
        - بيتشال (مش 1 ولا 0): مفيش حسم واضح خلال الأفق — حالة غامضة
          مالهاش لازمة تتعلم منها.
        ده بيخلي هدف التدريب مطابق تماماً لآلية التداول الفعلية، مش
        مقياس منفصل عنها.
        """
        df = df.copy()
        c = df["close"].values
        h = df["high"].values
        l = df["low"].values
        atr = df["atr"].values
        n = len(df)
        target = np.full(n, -1, dtype=int)

        for i in range(n - 1):
            a = atr[i]
            if np.isnan(a) or a <= 0:
                continue
            entry = c[i]
            tp = entry + atr_tp_mult * a
            sl = entry - atr_sl_mult * a
            end = min(i + 1 + horizon, n)
            for j in range(i + 1, end):
                if h[j] >= tp:
                    target[i] = 1
                    break
                if l[j] <= sl:
                    target[i] = 0
                    break

        df["target"] = target
        df["future_close"] = df["close"].shift(-horizon)
        df["pct_change"] = (df["future_close"] - df["close"]) / df["close"]
        n_before = len(df)
        df = df[df["target"] != -1].copy()
        n_after = len(df)
        logging.info(f"Triple-barrier target: اتشال {n_before - n_after} صف غامض "
                      f"(مفيش TP ولا SL اتحقق خلال {horizon} شمعة) من أصل {n_before} "
                      f"({(n_before-n_after)/max(n_before,1)*100:.1f}%) — الباقي حسم واضح")
        df["target"] = df["target"].astype(int)
        return df

    def fit_scaler(self, X):
        self.scaler.fit(X)
        self.scaler_fitted = True
        if isinstance(X, pd.DataFrame):
            self.feature_columns = list(X.columns)

    def transform(self, X):
        if not self.scaler_fitted:
            raise RuntimeError("الـ scaler لازم يتدرب أولاً (fit_scaler)")
        if isinstance(X, pd.DataFrame):
            X = X[self.feature_columns]
        return self.scaler.transform(X)

    def fit_transform(self, X):
        self.fit_scaler(X)
        return self.transform(X)


# ══════════════════════════════════════════════════════════════
# MULTI-TIMEFRAME BUILDER — HTF bias مبني على بيانات HTF مسحوبة
# مباشرة (native)، مش resample من فريم أصغر — يضمن عمق تاريخي حقيقي.
# ══════════════════════════════════════════════════════════════
class MultiTimeframeBuilder:
    HTF_FEATURES = [
        "market_structure_trend", "trend_filter", "pd_position",
        "is_premium", "is_discount", "atr_pct", "bos_up", "bos_down",
        "choch_bullish", "choch_bearish",
    ]

    def __init__(self):
        self._htf_cache = {}

    def get_htf_features(self, htf_key):
        if htf_key in self._htf_cache:
            return self._htf_cache[htf_key]
        if htf_key not in TIMEFRAME_CONFIG:
            return None
        cfg = TIMEFRAME_CONFIG[htf_key]
        df_htf_raw = get_or_fetch_timeframe_data(htf_key, cfg)
        if df_htf_raw.empty:
            return None
        fe_htf = SMCFeatureEngineer()
        df_htf_feat = fe_htf.create_features(df_htf_raw, cfg, shift=True)
        self._htf_cache[htf_key] = df_htf_feat
        return df_htf_feat

    def merge_htf_bias(self, df_ltf, larger_tf_key, label):
        if not larger_tf_key:
            return df_ltf
        df_htf = self.get_htf_features(larger_tf_key)
        if df_htf is None:
            logging.warning(f"تعذر جلب HTF ({larger_tf_key}) لدمجه مع {label}")
            return df_ltf

        available = [c for c in self.HTF_FEATURES if c in df_htf.columns]
        df_htf_sel = df_htf[["datetime"] + available].copy()
        df_htf_sel.columns = ["datetime"] + [f"htf_{label}_{c}" for c in available]
        df_htf_sel = df_htf_sel.sort_values("datetime")

        # merge_asof backward → لا lookahead: كل شمعة LTF تاخد آخر HTF قبلها فقط
        df_merged = pd.merge_asof(
            df_ltf.sort_values("datetime"), df_htf_sel,
            on="datetime", direction="backward",
        )
        return df_merged.ffill().fillna(0)


class GoldSilverRatioProvider:
    """
    نسبة الدهب/الفضة (Gold/Silver Ratio - GSR) — مقياس كلاسيكي خاص
    بمعدني الثمينة: كام أوقية فضة تساوي أوقية دهب. تاريخياً بيميل
    يرتد لمتوسطه على المدى المتوسط (mean-reversion) — مش موجود في موديل
    الدهب الأصلي لأنه فيتشور علاقة بين معدنين، مالوش معنى مماثل لموديل
    دهب مستقل. بنجيب سلسلة الدهب اليومية *مرة واحدة* بس (كاش) ونعمل
    merge_asof (backward) مع كل فريم — الإشارة دي بطيئة/macro أصلاً،
    مفيش داعي لدقة أعلى من يومي حتى لفريم الـ 5min.

    ⚠️ منع تسريب: بنستخدم إغلاق الدهب من *اليوم السابق* فقط (shift(1))
    بغض النظر عن توقيت الشمعة اليومية الفعلي عند مزود البيانات — عشان
    نضمن مفيش أي احتمال نستخدم سعر دهب لسه "لم يُغلق" وقت شمعة الفضة.
    """
    def __init__(self, years=20):
        self._gold_daily = None
        self._years = years

    def _load_gold_daily(self):
        if self._gold_daily is not None:
            return self._gold_daily
        df_gold = fetch_native_history("1day", self._years, symbol="XAU/USD")
        if df_gold.empty:
            logging.warning("تعذر جلب بيانات الدهب اليومية لحساب GSR — الفيتشور ده هيتشال")
            self._gold_daily = pd.DataFrame()
            return self._gold_daily
        df_gold = df_gold[["datetime", "close"]].rename(columns={"close": "gold_close"})
        df_gold = df_gold.sort_values("datetime").reset_index(drop=True)
        df_gold["gold_close"] = df_gold["gold_close"].shift(1)  # يوم سابق فقط — لا تسريب
        df_gold.dropna(subset=["gold_close"], inplace=True)
        self._gold_daily = df_gold
        return df_gold

    def merge_ratio(self, df_ltf):
        df_gold = self._load_gold_daily()
        if df_gold.empty:
            return df_ltf
        df_merged = pd.merge_asof(
            df_ltf.sort_values("datetime"), df_gold,
            on="datetime", direction="backward",
        )
        df_merged["gold_close"] = df_merged["gold_close"].ffill()
        gsr = df_merged["gold_close"] / df_merged["close"].replace(0, np.nan)
        df_merged["gold_silver_ratio"] = gsr.ffill().fillna(0)
        roll_mean = df_merged["gold_silver_ratio"].rolling(90, min_periods=20).mean()
        roll_std = df_merged["gold_silver_ratio"].rolling(90, min_periods=20).std()
        df_merged["gsr_zscore"] = (
            (df_merged["gold_silver_ratio"] - roll_mean) / roll_std.replace(0, np.nan)
        ).fillna(0).clip(-4, 4)
        df_merged["gsr_change_20"] = df_merged["gold_silver_ratio"].pct_change(20).fillna(0)
        df_merged.drop(columns=["gold_close"], inplace=True)
        return df_merged.reset_index(drop=True)


def _wilson_ci(successes, n, z=1.96):
    """
    فاصل ثقة Wilson لنسبة النجاح (Win Rate) — بيدي صورة صادقة لعدم اليقين
    لما عدد الصفقات قليل. مثال: 2 من 2 صفقة رابحة = "Win Rate 100%" ظاهرياً،
    لكن الفاصل الحقيقي بيبقى واسع جداً (تقريباً 20%-100%) — يعني الرقم
    غير موثوق إحصائياً برغم إنه شكله ممتاز.
    """
    if n == 0:
        return [0.0, 0.0]
    p = successes / n
    denom = 1 + z ** 2 / n
    center = (p + z ** 2 / (2 * n)) / denom
    margin = (z * ((p * (1 - p) / n + z ** 2 / (4 * n ** 2)) ** 0.5)) / denom
    return [round(max(0, (center - margin) * 100), 1), round(min(100, (center + margin) * 100), 1)]


# ══════════════════════════════════════════════════════════════
# BACKTESTER (Vectorized predict, loop فقط لمنطق الصفقات)
# ══════════════════════════════════════════════════════════════
class Backtester:
    def __init__(self, initial_capital=10000, spread_bps=INSTRUMENT_SPREAD_BPS,
                 slippage=INSTRUMENT_SLIPPAGE):
        self.initial_capital = initial_capital
        self.spread_bps = spread_bps / 10000
        self.slippage = slippage

    def _exec_price(self, price, is_buy):
        if is_buy:
            return price * (1 + self.spread_bps) * (1 + self.slippage)
        return price * (1 - self.spread_bps) * (1 - self.slippage)

    def run(self, df, model, fe, feature_columns, config, use_fold_thresholds=False):
        capital, positions, trades = self.initial_capital, [], []
        missing = [c for c in feature_columns + ["close", "atr"] if c not in df.columns]
        if missing:
            logging.error(f"أعمدة مفقودة في الباك تيست: {missing}")
            return {}

        buy_th = config.get("fold_buy_threshold" if use_fold_thresholds else "buy_threshold")
        sell_th = config.get("fold_sell_threshold" if use_fold_thresholds else "sell_threshold")
        max_daily = config.get("max_trades_per_day")
        daily_counts = {}

        X_scaled = fe.transform(df[feature_columns].copy())
        all_proba = model.predict_proba(X_scaled)[:, 1]

        # تشخيص: لو مفيش أي احتمال هيعدي العتبة أصلاً، سجّلها كـ DEBUG بس —
        # دي حالة طبيعية بتحصل في أول محاولة بالعتبة الثابتة قبل ما نجرب
        # العتبة الديناميكية في expanding_walk_forward، مش خطأ فعلي. لو
        # المحاولة الديناميكية كمان فشلت، هيتسجل WARNING واضح في مكانه.
        pct_above_buy = float((all_proba > buy_th).mean() * 100)
        pct_below_sell = float((all_proba < sell_th).mean() * 100)
        if pct_above_buy == 0 and pct_below_sell == 0:
            q = np.percentile(all_proba, [1, 25, 50, 75, 99])
            logging.debug(
                f"لا احتمال واحد عدّى buy_th={buy_th} أو نزل تحت sell_th={sell_th}. "
                f"توزيع الاحتمالات الفعلي — p1:{q[0]:.3f} p25:{q[1]:.3f} p50:{q[2]:.3f} "
                f"p75:{q[3]:.3f} p99:{q[4]:.3f}."
            )

        closes = df["close"].values
        atrs = df["atr"].values
        datetimes = df["datetime"].values if "datetime" in df.columns else None
        atr_sl = config.get("atr_sl_mult", 1.5)
        atr_tp = config.get("atr_tp_mult", 2.5)

        for i in range(len(df)):
            p, atr, prob = closes[i], atrs[i], all_proba[i]
            if np.isnan(atr) or atr <= 0:
                continue

            # إدارة الصفقة المفتوحة (لونج أو شورت)
            # ── باگ تم إصلاحه: النسخة الأصلية كانت بتتحقق فقط من stop_loss/
            # take_profit بمنطق اللونج (p<=sl / p>=tp) بغض النظر عن اتجاه
            # الصفقة، وكانت بتفتح BUY بس أبداً — SELL signal كان بيتحسب
            # لكنه محدش كان بيفتحله صفقة شورت فعلياً. ده معناه إن
            # sell_threshold (اللي بيتعاير ويتبحث عليه في كل مكان في
            # الكود) كان مالوش أي تأثير فعلي على أي نتيجة باك-تيست —
            # زيه زي عدم وجوده. الإصلاح: صفقة الشورت بتتفتح وتتقفل بمنطق
            # معكوس صح.
            if positions:
                pos = positions[0]
                exit_price, hit = None, None
                if pos["direction"] == "long":
                    if p <= pos["stop_loss"]:
                        exit_price, hit = pos["stop_loss"], "SL"
                    elif p >= pos["take_profit"]:
                        exit_price, hit = pos["take_profit"], "TP"
                else:  # short
                    if p >= pos["stop_loss"]:
                        exit_price, hit = pos["stop_loss"], "SL"
                    elif p <= pos["take_profit"]:
                        exit_price, hit = pos["take_profit"], "TP"
                if exit_price is not None:
                    if pos["direction"] == "long":
                        pnl = (exit_price - pos["entry_price"]) * pos["units"]
                        capital += pos["units"] * exit_price
                    else:
                        pnl = (pos["entry_price"] - exit_price) * pos["units"]
                        capital -= pos["units"] * exit_price
                    trades.append({"pnl": pnl, "exit_reason": hit, "direction": pos["direction"]})
                    positions = []

            signal = "HOLD"
            if prob > buy_th:
                signal = "BUY"
            elif prob < sell_th:
                signal = "SELL"

            if signal in ("BUY", "SELL") and max_daily and datetimes is not None:
                date_key = str(datetimes[i])[:10]
                if daily_counts.get(date_key, 0) >= max_daily:
                    signal = "HOLD"

            if signal == "BUY" and not positions:
                ep = self._exec_price(p, True)
                units = capital * 0.1 / ep
                positions.append({
                    "direction": "long", "entry_price": ep, "units": units,
                    "stop_loss": ep - atr_sl * atr,
                    "take_profit": ep + atr_tp * atr,
                })
                capital -= units * ep
                if max_daily and datetimes is not None:
                    date_key = str(datetimes[i])[:10]
                    daily_counts[date_key] = daily_counts.get(date_key, 0) + 1
            elif signal == "SELL" and not positions:
                ep = self._exec_price(p, False)
                units = capital * 0.1 / ep
                positions.append({
                    "direction": "short", "entry_price": ep, "units": units,
                    "stop_loss": ep + atr_sl * atr,
                    "take_profit": ep - atr_tp * atr,
                })
                capital += units * ep
                if max_daily and datetimes is not None:
                    date_key = str(datetimes[i])[:10]
                    daily_counts[date_key] = daily_counts.get(date_key, 0) + 1

        if not trades:
            return {}

        pnls = np.array([t["pnl"] for t in trades])
        wins = pnls[pnls > 0]
        losses = pnls[pnls <= 0]
        wr = (len(wins) / len(pnls)) * 100 if len(pnls) else 0
        gross_profit = wins.sum() if len(wins) else 0
        gross_loss = abs(losses.sum()) if len(losses) else 0
        pf = gross_profit / gross_loss if gross_loss > 0 else (gross_profit if gross_profit > 0 else 0)
        equity_curve = self.initial_capital + np.cumsum(pnls)
        running_max = np.maximum.accumulate(equity_curve)
        dd = ((equity_curve - running_max) / running_max * 100).min() if len(equity_curve) else 0
        aw = wins.mean() if len(wins) else 0
        al = abs(losses.mean()) if len(losses) else 0

        return {
            "total_trades": len(trades),
            "win_rate": round(wr, 2),
            "win_rate_ci_95": _wilson_ci(len(wins), len(pnls)),
            "profit_factor": round(pf, 2),
            "total_return_pct": round((equity_curve[-1] - self.initial_capital) / self.initial_capital * 100, 2),
            "max_drawdown_pct": round(dd, 2),
            "avg_win": round(aw, 2),
            "avg_loss": round(al, 2),
            "expectancy": round((wr / 100 * aw) - ((1 - wr / 100) * al), 2) if al else 0,
            "statistically_reliable": len(trades) >= config.get("min_reliable_trades", 15),
        }


# ══════════════════════════════════════════════════════════════
# WALK-FORWARD TRAINING — بدون Data Leakage
# ══════════════════════════════════════════════════════════════
def expanding_walk_forward(X, y, df_full, config, n_splits=5, purge_bars=0):
    """
    Expanding window + purge: كل fold بيتدرب على كل البيانات اللي قبله،
    وبيتقيّم على القطعة اللي بعده، مع `purge_bars` فاصل بين التدريب
    والاختبار عشان نمنع أي تسريب ناتج عن الـ horizon (تراكب الأهداف).
    """
    n = len(X)
    fold_size = n // (n_splits + 1)
    fold_metrics, best_model, best_fe, best_score = [], None, None, -np.inf
    # fallback ثانوي: أفضل فولد متاح حتى لو ملوش عدد صفقات كافي — بدل
    # ما نختار "آخر فولد" عشوائياً لو محدش عدى الحد الأدنى.
    fallback_model, fallback_fe, fallback_score = None, None, -np.inf
    best_n_trees, fallback_n_trees = None, None
    MIN_FOLD_TRADES = 3
    model = None

    for fold in range(n_splits):
        train_end = fold_size * (fold + 1)
        test_start = train_end + purge_bars
        test_end = min(test_start + fold_size, n)
        if test_start >= test_end:
            continue

        X_tr, y_tr = X.iloc[:train_end], y.iloc[:train_end]
        X_te, y_te = X.iloc[test_start:test_end], y.iloc[test_start:test_end]
        if len(X_tr) < 100 or len(X_te) < 20:
            continue

        neg, pos = (y_tr == 0).sum(), (y_tr == 1).sum()
        spw = neg / pos if pos > 0 else 1

        # ── فصل الـ early-stopping عن فولد الاختبار الحقيقي ──
        # كان الكود بيستخدم X_te/y_te (فولد الاختبار) كـ eval_set، يعني
        # عدد الأشجار بيتحدد بناءً على نفس البيانات اللي هنقيّم عليها —
        # ده بيخلي التقييم متأثر بقرار تدريبي اتخد وهو شايف نتيجة الاختبار.
        # الصح: نقتطع validation داخلي من آخر جزء من التدريب نفسه (مش من
        # فولد الاختبار)، ونسيب فولد الاختبار "أعمى" تماماً لحد التقييم.
        val_size = max(int(len(X_tr) * 0.15), 50)
        X_tr_fit, y_tr_fit = X_tr.iloc[:-val_size], y_tr.iloc[:-val_size]
        X_tr_val, y_tr_val = X_tr.iloc[-val_size:], y_tr.iloc[-val_size:]

        fe_fold = SMCFeatureEngineer()
        fe_fold.fit_scaler(X_tr_fit)
        X_tr_fit_s = fe_fold.transform(X_tr_fit)
        X_tr_val_s = fe_fold.transform(X_tr_val)
        X_te_s = fe_fold.transform(X_te)

        model = xgb.XGBClassifier(
            objective="binary:logistic", n_estimators=800, max_depth=4,
            learning_rate=0.03, subsample=0.7, colsample_bytree=0.6,
            reg_alpha=0.3, reg_lambda=2.5, min_child_weight=5, scale_pos_weight=spw,
            random_state=42, tree_method="hist", verbosity=0,
            eval_metric="logloss", early_stopping_rounds=40,
        )
        model.fit(X_tr_fit_s, y_tr_fit, eval_set=[(X_tr_val_s, y_tr_val)], verbose=False)

        # تشخيص: كام شجرة فعلياً استخدمت قبل ما الـ early stopping يوقف.
        # لو الرقم صغير جداً (مثلاً <20 من أصل 800)، معناها الموديل مالقاش
        # إشارة تستاهل يكمل تدريب عليها — ده مؤشر ضعف edge حقيقي مش باگ.
        n_trees_used = getattr(model, "best_iteration", None)
        logging.info(f"    Fold {fold+1}: عدد الأشجار المستخدمة فعلياً = "
                      f"{n_trees_used if n_trees_used is not None else model.n_estimators} "
                      f"(من أصل 800)")

        y_pred = model.predict(X_te_s)
        acc = accuracy_score(y_te, y_pred)
        prec = precision_score(y_te, y_pred, zero_division=0)
        rec = recall_score(y_te, y_pred, zero_division=0)

        df_test = df_full.iloc[test_start:test_end].copy()
        bt = Backtester().run(df_test, model, fe_fold, fe_fold.feature_columns,
                               config, use_fold_thresholds=True)

        # لو العتبات الثابتة في الكونفيج مانتجتش أي صفقة في الفولد ده،
        # جرّب عتبة ديناميكية (كوانتايل من توزيع احتمالات الفولد نفسه)
        # لغرض التقييم/الاختيار فقط — من غير ما نلمس عتبات الإنتاج.
        used_dynamic_th = False
        if not bt or bt.get("total_trades", 0) == 0:
            X_te_scaled = fe_fold.transform(X_te)
            proba_te = model.predict_proba(X_te_scaled)[:, 1]
            dyn_buy = float(np.percentile(proba_te, 85))
            dyn_sell = float(np.percentile(proba_te, 15))
            dyn_config = dict(config)
            dyn_config["fold_buy_threshold"] = dyn_buy
            dyn_config["fold_sell_threshold"] = dyn_sell
            bt_dyn = Backtester().run(df_test, model, fe_fold, fe_fold.feature_columns,
                                       dyn_config, use_fold_thresholds=True)
            if bt_dyn and bt_dyn.get("total_trades", 0) > 0:
                bt = bt_dyn
                used_dynamic_th = True
                logging.info(f"    Fold {fold+1}: العتبة الثابتة مافيهاش صفقات → "
                              f"استخدمنا عتبة ديناميكية (buy>{dyn_buy:.3f}, sell<{dyn_sell:.3f}) "
                              f"للتقييم فقط")
            else:
                logging.warning(
                    f"    Fold {fold+1}: مفيش صفقات حتى بالعتبة الديناميكية (buy>{dyn_buy:.3f}, "
                    f"sell<{dyn_sell:.3f}) — الإشارة ضعيفة جداً في الفولد ده تحديداً"
                )

        logging.info(f"  Fold {fold+1}/{n_splits} — Acc:{acc:.2%} Prec:{prec:.2%} Rec:{rec:.2%}")
        if bt:
            logging.info(f"    BT{'(dyn)' if used_dynamic_th else ''} — WR:{bt['win_rate']}% "
                          f"PF:{bt['profit_factor']} Trades:{bt['total_trades']} "
                          f"Ret:{bt['total_return_pct']}%")

        fold_metrics.append({"fold": fold + 1, "accuracy": acc, "precision": prec,
                              "recall": rec, "backtest": bt,
                              "used_dynamic_threshold": used_dynamic_th})

        if bt and bt.get("total_trades", 0) > 0:
            score = bt["profit_factor"] * np.log1p(bt["total_trades"])
            # يدخل الترشيح الأساسي فقط لو عدّى الحد الأدنى المعقول للصفقات
            if bt["total_trades"] >= MIN_FOLD_TRADES and score > best_score:
                best_score, best_model, best_fe = score, model, fe_fold
                best_n_trees = n_trees_used
            # وفي كل الأحوال نتابع أفضل فولد حتى لو تحت الحد — fallback واعي
            if score > fallback_score:
                fallback_score, fallback_model, fallback_fe = score, model, fe_fold
                fallback_n_trees = n_trees_used

    if best_model is None:
        if fallback_model is not None:
            logging.warning(
                f"لا فولد وصل لحد {MIN_FOLD_TRADES} صفقات — fallback لأفضل فولد "
                f"متاح (score={fallback_score:.3f}) بدل آخر فولد عشوائياً"
            )
            best_model, best_fe, best_n_trees = fallback_model, fallback_fe, fallback_n_trees
        elif model is not None:
            logging.warning("كل الفولدات بلا صفقات إطلاقاً — fallback لآخر موديل تدرب "
                             "(راجع الـ thresholds، ممكن تكون عالية جداً لحجم الداتا الحالي)")
            best_model, best_fe, best_n_trees = model, fe_fold, n_trees_used

    final_n_trees = best_n_trees
    if final_n_trees is None or final_n_trees < 10:
        final_n_trees = None  # نرجع للـ default (800) لو الرقم غير موثوق فيه
    return fold_metrics, best_model, best_fe, final_n_trees


def train_smc_model_for_timeframe(tf_key, n_splits=5, mtf_builder=None, force_refresh=False,
                                   target_mode=None, gsr_provider=None):
    config = TIMEFRAME_CONFIG[tf_key]
    logging.info("=" * 60)
    logging.info(f"SMC Walk-Forward — فريم {tf_key}")
    logging.info("=" * 60)

    df_raw = get_or_fetch_timeframe_data(tf_key, config, force_refresh=force_refresh)
    if df_raw.empty or len(df_raw) < config["min_candles"]:
        logging.error(f"بيانات غير كافية لفريم {tf_key} ({len(df_raw)} شمعة)")
        return None

    fe = SMCFeatureEngineer()
    df_feat = fe.create_features(df_raw, config, shift=True)
    effective_mode = target_mode or ("barrier" if config.get("use_barrier_target", True) else "atr_filter")
    if effective_mode == "barrier":
        df_target = fe.create_target_triple_barrier(
            df_feat, horizon=config["horizon"],
            atr_sl_mult=config["atr_sl_mult"], atr_tp_mult=config["atr_tp_mult"],
        )
    else:
        df_target = fe.create_target(df_feat, horizon=config["horizon"],
                                      min_atr_mult=config.get("target_atr_mult"))
    logging.info(f"[{tf_key}] طريقة تصنيف الهدف المستخدمة: {effective_mode}")
    if df_target.empty:
        logging.error("لا بيانات بعد إنشاء الهدف")
        return None

    if mtf_builder and config.get("larger_tf"):
        df_target = mtf_builder.merge_htf_bias(df_target, config["larger_tf"], tf_key)

        # ─── تقوية Confluence بمحاذاة HTF: إعداد صاعد على الفريم الحالي
        # مع Bias صاعد على الفريم الأعلى أقوى بكتير من نفس الإعداد وسط
        # اتجاه HTF معاكس. ده أهم فيتشور في منهجية ICT (top-down analysis)
        # ومكانش مُفعّل قبل كده رغم إن الداتا كانت متاحة أصلاً. ───
        htf_trend_col = f"htf_{tf_key}_market_structure_trend"
        if htf_trend_col in df_target.columns and "smc_confluence_bull" in df_target.columns:
            htf_trend = df_target[htf_trend_col]
            htf_bull_align = (htf_trend == 1).astype(int)
            htf_bear_align = (htf_trend == -1).astype(int)
            df_target["smc_confluence_bull_htf"] = df_target["smc_confluence_bull"] * (
                1.0 + 0.5 * htf_bull_align - 0.3 * htf_bear_align
            ).clip(0.2, 1.5)
            df_target["smc_confluence_bear_htf"] = df_target["smc_confluence_bear"] * (
                1.0 + 0.5 * htf_bear_align - 0.3 * htf_bull_align
            ).clip(0.2, 1.5)
            df_target["smc_confluence_net_htf"] = (
                df_target["smc_confluence_bull_htf"] - df_target["smc_confluence_bear_htf"]
            )
            # Higher Timeframe Confluence Filter: فلتر ثنائي واضح — هل
            # الإعداد الحالي (bull أو bear) متفق مع اتجاه الفريم الأعلى؟
            # ده فيتشور "قرار" جاهز، مش مقياس مستمر، يسهّل على الموديل
            # يستخدمه كبوابة مباشرة (زي قاعدة ICT: "متفتحش صفقة عكس الـ
            # 1H/4H bias").
            strong_bull_setup = df_target["smc_confluence_bull"] >= 0.5
            strong_bear_setup = df_target["smc_confluence_bear"] >= 0.5
            df_target["htf_confluence_filter_bull_pass"] = (
                strong_bull_setup & (htf_bull_align == 1)
            ).astype(int)
            df_target["htf_confluence_filter_bear_pass"] = (
                strong_bear_setup & (htf_bear_align == 1)
            ).astype(int)

    if gsr_provider is not None:
        df_target = gsr_provider.merge_ratio(df_target)

    exclude_final = ["datetime", "open", "high", "low", "close", "volume",
                      "future_close", "pct_change", "target"]
    fe.feature_columns = [c for c in df_target.columns if c not in exclude_final]
    df_target = df_target.reset_index(drop=True)

    # ─── HOLDOUT معزول بالكامل من أول لحظة — مع تكييف حجمه على كمية الداتا ───
    # 15% ثابت بيدي holdout صغير جداً على الفريمات العالية (4h/1d/1W) —
    # 14 صفقة أو أقل، مش كافي إحصائياً. لو الداتا كافية (≥1000 صف)،
    # بنكبّر الـ holdout لـ 20% عشان التقييم النهائي يبقى موثوق.
    adaptive_holdout_pct = HOLDOUT_PCT
    if len(df_target) >= 1000:
        adaptive_holdout_pct = 0.20
    holdout_start = int(len(df_target) * (1 - adaptive_holdout_pct))
    df_train_pool = df_target.iloc[:holdout_start].copy()
    df_holdout = df_target.iloc[holdout_start:].copy()
    logging.info(
        f"Train pool: {len(df_train_pool)} | Holdout ({int(adaptive_holdout_pct*100)}% - "
        f"لم يُرَ إطلاقاً أثناء التدريب): {len(df_holdout)}"
    )

    X_pool = df_train_pool[fe.feature_columns]
    y_pool = df_train_pool["target"]

    purge_bars = config["horizon"] * 2  # يمنع تسريب overlap بسبب الـ horizon
    fold_metrics, best_model, best_fe, recommended_n_trees = expanding_walk_forward(
        X_pool, y_pool, df_train_pool, config, n_splits=n_splits, purge_bars=purge_bars
    )
    if best_model is None:
        logging.error(f"فشل التدريب لفريم {tf_key} — لا فولد صالح")
        return None
    n_trees_final = recommended_n_trees or best_model.n_estimators
    logging.info(f"عدد الأشجار المستخدم لموديلي التقييم/الإنتاج: {n_trees_final}")

    # ─── تقليل عدد الفيتشورز حسب الأهمية الفعلية (Feature Importance) ───
    # المشكلة اللي بتظهر فعلياً: عدد الفيتشورز الكبير نسبة لحجم العينة
    # المتاح (خصوصاً الفريمات الأعلى زي 1d/1W) بيخلي الموديل عرضة لـ
    # overfitting — يحفظ نويز الشريحة بدل ما يتعلم نمط حقيقي معمّم.
    # الحل: ناخد ترتيب الأهمية من موديل الـ CV (best_model، اتدرب على
    # بيانات out-of-fold حقيقية)، ونختار أفضل عدد محدود منها بس لباقي
    # المراحل. كل صف يستحمل ~8 فيتشورز فقط بأمان.
    if best_model is not None and hasattr(best_model, "feature_importances_"):
        importances = best_model.feature_importances_
        feat_imp_pairs = sorted(zip(X_pool.columns, importances), key=lambda x: x[1], reverse=True)
        max_by_sample_size = max(15, len(X_pool) // 8)
        max_features = min(45, max_by_sample_size, len(feat_imp_pairs))
        selected_features = [f for f, imp in feat_imp_pairs[:max_features] if imp > 0]
        if len(selected_features) < 10:
            selected_features = [f for f, _ in feat_imp_pairs[:min(15, len(feat_imp_pairs))]]
        if len(selected_features) < len(X_pool.columns):
            logging.info(
                f"[{tf_key}] تقليل الفيتشورز من {len(X_pool.columns)} إلى "
                f"{len(selected_features)} (أهم فيتشورز بالـ Feature Importance من فولدات "
                f"الـ CV الحقيقية) — تقليل خطر الـ overfitting مع حجم العينة المتاح "
                f"({len(X_pool)} صف)."
            )
        fe.feature_columns = selected_features
        X_pool = X_pool[selected_features]
        y_pool = df_train_pool["target"]

    # ─── Final (evaluation) model: يتدرب على كل الـ train_pool فقط ───
    fe_eval = SMCFeatureEngineer()
    fe_eval.fit_scaler(X_pool)
    X_pool_s = fe_eval.transform(X_pool)
    neg, pos = (y_pool == 0).sum(), (y_pool == 1).sum()

    eval_model = xgb.XGBClassifier(
        objective="binary:logistic", n_estimators=n_trees_final,
        max_depth=best_model.max_depth, learning_rate=best_model.learning_rate,
        subsample=best_model.subsample, colsample_bytree=best_model.colsample_bytree,
        reg_alpha=best_model.reg_alpha, reg_lambda=best_model.reg_lambda,
        min_child_weight=best_model.min_child_weight,
        scale_pos_weight=neg / pos if pos else 1, random_state=42,
        tree_method="hist", verbosity=0, eval_metric="logloss",
    )
    eval_model.fit(X_pool_s, y_pool)

    # ─── معايرة العتبات تلقائياً بدل الأرقام الثابتة المخمَّنة في الكونفيج ───
    # المشكلة الأصلية: buy_threshold/sell_threshold في TIMEFRAME_CONFIG أرقام
    # ثابتة اتحطت تخمين، ومفيش ضمان إنها تتناسب مع التوزيع الفعلي لاحتمالات
    # الموديل. هنا بنقتطع "شريحة معايرة" (آخر 15% من train_pool، قبل
    # الـ holdout) وندرب موديل مؤقت على الباقي بس، نجيب احتمالاته على
    # الشريحة دي (لم يرها أثناء تدريبه)، ونحسب عتبات البيع/الشراء منها
    # كـ percentile فعلي — مش تخمين.
    # ─── شريحة المعايرة مقسومة لنصفين: "بحث" (الأقدم) و"تأكيد" (الأحدث) ───
    # قبل قبول أي عتبة، لازم تثبت ربحيتها على نصفين منفصلين زمنياً من
    # شريحة المعايرة (نصف قديم للبحث + نصف أحدث للتأكيد). ده فلتر
    # إضافي بيمسك overfitting قبل ما نوصل للـ holdout الحقيقي أصلاً.
    calib_size = max(int(len(X_pool) * 0.15), 100)
    X_calib_train, y_calib_train = X_pool.iloc[:-calib_size], y_pool.iloc[:-calib_size]
    X_calib_test = X_pool.iloc[-calib_size:]
    df_calib_test = df_train_pool.iloc[-calib_size:]
    valid_atr_rows = int((df_calib_test["atr"].notna() & (df_calib_test["atr"] > 0)).sum())

    MIN_CONFIRM_ROWS = 30
    _half = len(X_calib_test) // 2
    X_calib_search, df_calib_search = X_calib_test.iloc[:_half], df_calib_test.iloc[:_half]
    X_calib_confirm, df_calib_confirm = X_calib_test.iloc[_half:], df_calib_test.iloc[_half:]
    has_confirm_slice = len(X_calib_search) >= MIN_CONFIRM_ROWS and len(X_calib_confirm) >= MIN_CONFIRM_ROWS
    logging.info(f"[{tf_key}] حجم شريحة المعايرة: {len(df_calib_test)} صف، منهم "
                 f"{valid_atr_rows} صف بـ ATR صالح (قابل فعلياً لفتح صفقة). "
                 f"إجمالي train_pool: {len(df_train_pool)}, إجمالي الداتا الخام "
                 f"بعد الفلترة: {len(df_target)}")

    calibrated_buy_th, calibrated_sell_th = config["buy_threshold"], config["sell_threshold"]
    calib_search_log = []
    no_viable_threshold = False
    if len(X_calib_train) <= 200 or len(X_calib_test) <= 50:
        logging.warning(
            f"[{tf_key}] بيانات قليلة جداً لتشغيل بحث المعايرة (calib_train="
            f"{len(X_calib_train)}, calib_test={len(X_calib_test)}) — هنستخدم عتبات "
            f"الكونفيج الثابتة مباشرة على الـ holdout بدون بحث. الحل الحقيقي هنا "
            f"محتاج تاريخ بيانات أطول لهذا الفريم."
        )
    if len(X_calib_train) > 200 and len(X_calib_test) > 50:
        # فصل جزء صغير من calib_train نفسه للـ early-stopping، عشان موديل
        # المعايرة متعملوش overfit (زي ما حصل قبل كده لما درّبناه بعدد
        # أشجار ثابت كبير من غير أي إيقاف مبكر — ده بيطلع عتبات متطرفة
        # وهمية زي buy=0.99 مش عتبة حقيقية).
        cv_val_size = max(int(len(X_calib_train) * 0.15), 50)
        X_ct_fit, y_ct_fit = X_calib_train.iloc[:-cv_val_size], y_calib_train.iloc[:-cv_val_size]
        X_ct_val, y_ct_val = X_calib_train.iloc[-cv_val_size:], y_calib_train.iloc[-cv_val_size:]

        neg_c, pos_c = (y_ct_fit == 0).sum(), (y_ct_fit == 1).sum()
        fe_calib = SMCFeatureEngineer()
        fe_calib.fit_scaler(X_ct_fit)
        calib_model = xgb.XGBClassifier(
            objective="binary:logistic", n_estimators=800,
            max_depth=best_model.max_depth, learning_rate=best_model.learning_rate,
            subsample=best_model.subsample, colsample_bytree=best_model.colsample_bytree,
            reg_alpha=best_model.reg_alpha, reg_lambda=best_model.reg_lambda,
            min_child_weight=best_model.min_child_weight,
            scale_pos_weight=neg_c / pos_c if pos_c else 1, random_state=42,
            tree_method="hist", verbosity=0, eval_metric="logloss",
            early_stopping_rounds=40,
        )
        calib_model.fit(fe_calib.transform(X_ct_fit), y_ct_fit,
                         eval_set=[(fe_calib.transform(X_ct_val), y_ct_val)], verbose=False)
        calib_proba = calib_model.predict_proba(fe_calib.transform(X_calib_test))[:, 1]
        cq = np.percentile(calib_proba, [1, 25, 50, 75, 95, 99])
        logging.info(f"[{tf_key}] توزيع احتمالات المعايرة (n={len(calib_proba)}) — "
                      f"p1:{cq[0]:.3f} p25:{cq[1]:.3f} p50:{cq[2]:.3f} p75:{cq[3]:.3f} "
                      f"p95:{cq[4]:.3f} p99:{cq[5]:.3f}")

        # ─── بحث فعلي عن أفضل عتبة، مش مجرد percentile ثابت مخمَّن ───
        # المشكلة اللي ظهرت: تخفيف العتبة (نسبة مئوية أقل تشدداً) بيولّد
        # صفقات أكتر فعلاً، لكن مفيش ضمان إنها صفقات مربحة — ممكن تكون
        # كمية على حساب الجودة (زي ما حصل بالظبط في 1h/2h/4h: عدد صفقات
        # كويس لكن PF<1). الحل الصح: نجرب كذا percentile مرشح، ونحسب
        # فعلياً أداء كل واحد منهم على شريحة المعايرة (بالسعر الحقيقي عبر
        # الـ Backtester)، ونختار اللي بيحقق أعلى ربحية فعلية مش أكتر عدد
        # صفقات.
        # ─── رفع الحد الأدنى الإحصائي لعدد الصفقات المقبولة ───
        # المشكلة السابقة: كان ممكن يظهر "PF=2.92@3 صفقات" في اللوج
        # كمرشح جاد، رغم إن 3 صفقات مفيهاش أي دلالة إحصائية (ممكن صدفة).
        # الحد الجديد يضمن ما نأخذش قرار على أرقام غير موثوقة.
        min_trades_for_search = max(15, config.get("min_reliable_trades", 15) // 2)
        candidate_percentiles = [95, 92, 88, 84, 80, 76, 72]
        best_search_score = -np.inf
        if not has_confirm_slice:
            logging.info(
                f"[{tf_key}] شريحة المعايرة صغيرة لتقسيمها لنصفين (بحث/تأكيد) — "
                f"هنكتفي بالبحث على الشريحة كلها، من غير فحص تأكيد إضافي."
            )
        for pct in candidate_percentiles:
            # الحد الأدنى بقى قريب من 0.5 بدل 0.51 — الفلور القديم كان
            # بيمنع أي فريم ضعيف الإشارة من توليد صفقة إطلاقاً حتى لو
            # فيه ميزة صغيرة حقيقية، لأن التوزيع الفعلي كان بيفضل تحت
            # 0.51 دايماً فمفيش عتبة كانت هتتحقق مهما اخترنا percentile.
            cb = float(np.clip(np.percentile(calib_proba, pct), 0.502, 0.90))
            cs = float(np.clip(np.percentile(calib_proba, 100 - pct), 0.10, 0.498))
            if cb <= 0.5 or cs >= 0.5 or (cb - cs) <= 0.01:
                continue
            trial_config = dict(config)
            trial_config["buy_threshold"] = cb
            trial_config["sell_threshold"] = cs
            trial_bt = Backtester().run(df_calib_test, calib_model, fe_calib,
                                         fe_calib.feature_columns, trial_config,
                                         use_fold_thresholds=False)
            n_tr = trial_bt.get("total_trades", 0) if trial_bt else 0
            pf_tr = trial_bt.get("profit_factor", 0) if trial_bt else 0

            # ─── فحص التأكيد: هل نفس العتبة ربحية على نص زمني تاني؟ ───
            confirm_pf, confirm_tr, search_pf, search_tr = None, 0, None, 0
            passes_confirm_split = True
            if has_confirm_slice:
                search_bt = Backtester().run(df_calib_search, calib_model, fe_calib,
                                              fe_calib.feature_columns, trial_config,
                                              use_fold_thresholds=False)
                confirm_bt = Backtester().run(df_calib_confirm, calib_model, fe_calib,
                                               fe_calib.feature_columns, trial_config,
                                               use_fold_thresholds=False)
                search_pf = search_bt.get("profit_factor", 0) if search_bt else 0
                search_tr = search_bt.get("total_trades", 0) if search_bt else 0
                confirm_pf = confirm_bt.get("profit_factor", 0) if confirm_bt else 0
                confirm_tr = confirm_bt.get("total_trades", 0) if confirm_bt else 0
                min_half_trades = max(3, min_trades_for_search // 2)
                passes_confirm_split = (
                    search_tr >= min_half_trades and confirm_tr >= min_half_trades and
                    search_pf > 1.0 and confirm_pf > 1.0
                )

            insufficient_sample = n_tr < min_trades_for_search
            calib_search_log.append({
                "percentile": pct, "buy_th": round(cb, 3), "sell_th": round(cs, 3),
                "trades": n_tr, "profit_factor": pf_tr,
                "confirm_split_search_pf": round(search_pf, 2) if search_pf is not None else None,
                "confirm_split_confirm_pf": round(confirm_pf, 2) if confirm_pf is not None else None,
                "passes_confirm_split": passes_confirm_split,
                "insufficient_sample": insufficient_sample,
            })
            if n_tr >= min_trades_for_search and pf_tr > 1.05 and passes_confirm_split:
                # بوابة ربحية إجبارية: أي عتبة PF فيها ≤1.05 (خسرانة أو
                # breakeven تقريباً) بتتشال فوراً من المنافسة، مهما كان
                # عدد صفقاتها كبير. المشكلة اللي كانت موجودة قبل كده: PF=0.9
                # بـ50 صفقة كان بينافس ويكسب أحياناً على PF=3 بصفقتين، وده
                # غلط منطقياً — استراتيجية خسرانة تفضل خسرانة مهما تكررت.
                score = pf_tr * np.log1p(n_tr)
                if score > best_search_score:
                    best_search_score = score
                    calibrated_buy_th, calibrated_sell_th = cb, cs

        if best_search_score > -np.inf:
            search_summary = ", ".join(
                f"{c['percentile']}%→PF{c['profit_factor']}@{c['trades']}تج"
                + (" ⚠️عينة_صغيرة_متجاهلة" if c.get("insufficient_sample") else "")
                + (f"[تأكيد:بحث{c['confirm_split_search_pf']}/تأكيد{c['confirm_split_confirm_pf']}"
                   f"{'✅' if c['passes_confirm_split'] else '❌'}]"
                   if c.get("confirm_split_search_pf") is not None else "")
                for c in calib_search_log
            )
            logging.info(f"✅ عتبات معايرة عبر بحث فعلي على أداء حقيقي — "
                          f"buy={calibrated_buy_th:.3f} sell={calibrated_sell_th:.3f} "
                          f"(score={best_search_score:.3f}) — [{search_summary}]")
        else:
            # نتأكد لو العتبة الثابتة في الكونفيج نفسها بتعدي بوابة الربحية
            # على نفس شريحة المعايرة — لو برضو لأ، الفريم ده مالوش عتبة
            # مربحة معروفة حالياً، ولازم نقولها صراحة بدل ما نديه موديل
            # هيتباع كأنه جاهز للتداول المستقل.
            static_trial = dict(config)
            static_bt = Backtester().run(df_calib_test, calib_model, fe_calib,
                                          fe_calib.feature_columns, static_trial,
                                          use_fold_thresholds=False)
            static_pf = static_bt.get("profit_factor", 0) if static_bt else 0
            static_trades = static_bt.get("total_trades", 0) if static_bt else 0
            if static_pf > 1.05 and static_trades >= min_trades_for_search:
                logging.warning(
                    f"[{tf_key}] البحث مالقاش عتبة أفضل من الثابتة — هنستخدم "
                    f"القيم الثابتة في الكونفيج (PF={static_pf} على شريحة المعايرة)"
                )
            else:
                no_viable_threshold = True
                search_summary2 = ", ".join(
                    f"{c['percentile']}%→PF{c['profit_factor']}@{c['trades']}تج"
                    + (" ⚠️عينة_صغيرة_متجاهلة" if c.get("insufficient_sample") else "")
                    for c in calib_search_log
                )
                logging.warning(
                    f"[{tf_key}] ⚠️ لا توجد عتبة (لا من البحث ولا الثابتة) بتحقق "
                    f"ربحية حقيقية (PF>1.05) على شريحة المعايرة — [{search_summary2}, "
                    f"ثابتة→PF{static_pf}@{static_trades}تج]. الفريم ده مفيهوش edge "
                    f"حقيقي حالياً كاستراتيجية مستقلة بالفيتشورز الحالية — استخدمه "
                    f"كـ HTF context/bias بس، مش كقرار تداول قائم بذاته."
                )

    calibrated_config = dict(config)
    calibrated_config["buy_threshold"] = calibrated_buy_th
    calibrated_config["sell_threshold"] = calibrated_sell_th
    calibrated_config["viable_for_standalone_trading"] = not no_viable_threshold

    # ─── تقييم حقيقي على الـ holdout — الموديل لم يرَ هذه البيانات إطلاقاً ───
    holdout_bt = Backtester().run(
        df_holdout, eval_model, fe_eval, fe_eval.feature_columns, calibrated_config,
        use_fold_thresholds=False,
    )
    if holdout_bt:
        reliability_tag = "✅ موثوق إحصائياً" if holdout_bt["statistically_reliable"] else \
            f"⚠️ غير موثوق (أقل من {config.get('min_reliable_trades', 15)} صفقة)"
        logging.info(f"{'✅' if holdout_bt['statistically_reliable'] else '⚠️'} "
                      f"Holdout حقيقي (لا تسريب) — WR:{holdout_bt['win_rate']}% "
                      f"(CI95: {holdout_bt['win_rate_ci_95'][0]}-{holdout_bt['win_rate_ci_95'][1]}%) "
                      f"PF:{holdout_bt['profit_factor']} Trades:{holdout_bt['total_trades']} "
                      f"Ret:{holdout_bt['total_return_pct']}% DD:{holdout_bt['max_drawdown_pct']}% "
                      f"— {reliability_tag}")
        if not holdout_bt["statistically_reliable"]:
            logging.warning(
                f"[{tf_key}] عدد صفقات الـ holdout ({holdout_bt['total_trades']}) قليل جداً — "
                f"أي رقم PF أو Win Rate هنا (حتى لو شكله ممتاز زي PF عالي جداً) مش دليل "
                f"موثوق على أداء حقيقي. محتاج فترة تاريخية أطول أو تكرار الاختبار على "
                f"فترات مختلفة قبل ما نثق في الأرقام دي."
            )
    else:
        logging.warning(f"لا صفقات في الـ holdout لفريم {tf_key} حتى بعد المعايرة — "
                         f"الإشارة المتاحة ضعيفة جداً لهذا الفريم بالفيتشورز الحالية")

    # ─── اختبار استقرار عبر عدة نوافذ زمنية منفصلة — رقم واحد من holdout
    # واحد ممكن يكون حظ (كويس أو وحش)، مش دليل على edge حقيقي متكرر.
    # بدل ما نعيد اختبار eval_model على جزء من بياناته هو نفسه (ده كان
    # هيبقى تسريب داخلي — الموديل شاف الداتا دي وقت تدريبه)، بنستخدم
    # نتائج الفولدات في expanding_walk_forward اللي فعلاً كل واحد فيها
    # اتقيّم على بيانات لم يرها موديل الفولد ده بالذات. ده الفرق بين
    # "edge حقيقي متكرر عبر فترات مختلفة" و"صدفة إحصائية في فترة واحدة".
    fold_pfs = [
        min(fm["backtest"].get("profit_factor", 0), 10)  # قصّ القيم المتطرفة الناتجة
        for fm in fold_metrics if fm.get("backtest")       # عن عينات صغيرة جداً — عشان
    ]                                                        # متضخمش المتوسط بشكل مضلل
    if len(fold_pfs) >= 2:
        # المتوسط الحسابي لـ PF مضلل هنا لأنه غير محدود من فوق وبيتأثر
        # بشدة بفولد واحد "محظوظ" بعدد صفقات قليل (زي PF=53 من 16 صفقة
        # بس) حتى لو فولد تاني كان خسران بالكامل. الوسيط (median) ونسبة
        # الفولدات الرابحة مقاييس أصدق هنا.
        pf_median = float(np.median(fold_pfs))
        profitable_fold_ratio = sum(1 for p in fold_pfs if p > 1.0) / len(fold_pfs)
        consistent = profitable_fold_ratio >= 0.66 and pf_median > 1.1
        logging.info(
            f"{'✅' if consistent else '⚠️'} [{tf_key}] استقرار عبر {len(fold_pfs)} فولدات "
            f"CV (كلها out-of-sample حقيقياً) — PF لكل فولد (مقصوصة عند 10): "
            f"{[round(p,2) for p in fold_pfs]} (وسيط={pf_median:.2f}, فولدات رابحة: "
            f"{sum(1 for p in fold_pfs if p > 1.0)}/{len(fold_pfs)}) — "
            f"{'edge متسق نسبياً عبر فترات مختلفة' if consistent else 'الأداء متذبذب — مؤشر قوي على عدم وجود edge حقيقي مستقر، مش بس مشكلة عتبة أو معايرة'}"
        )
        calibrated_config["stability_check_pf_by_fold"] = [round(p, 2) for p in fold_pfs]
        calibrated_config["stability_consistent"] = consistent

    os.makedirs(MODELS_DIR, exist_ok=True)

    # ─── Production model: يتدرب على 100% (train_pool + holdout) للنشر الفعلي.
    #     ⚠️ تنبيه صريح: مقاييس الـ holdout أعلاه لا تنطبق على هذا الموديل
    #     لأنه رأى تلك البيانات أثناء التدريب. استخدمها فقط للنشر، ولإعادة
    #     التقييم اعتمد على holdout_bt الخاص بـ eval_model. ───
    X_full = df_target[fe.feature_columns]
    y_full = df_target["target"]
    fe_prod = SMCFeatureEngineer()
    fe_prod.fit_scaler(X_full)
    X_full_s = fe_prod.transform(X_full)
    neg_f, pos_f = (y_full == 0).sum(), (y_full == 1).sum()

    prod_model = xgb.XGBClassifier(
        objective="binary:logistic", n_estimators=n_trees_final,
        max_depth=best_model.max_depth, learning_rate=best_model.learning_rate,
        subsample=best_model.subsample, colsample_bytree=best_model.colsample_bytree,
        reg_alpha=best_model.reg_alpha, reg_lambda=best_model.reg_lambda,
        min_child_weight=best_model.min_child_weight,
        scale_pos_weight=neg_f / pos_f if pos_f else 1, random_state=42,
        tree_method="hist", verbosity=0, eval_metric="logloss",
    )
    prod_model.fit(X_full_s, y_full)

    # ─── Feature Importance: نشوف فعلياً أي فيتشورز بتأثر على قرار
    # الموديل — ده تشخيص أساسي: لو فيتشورز SMC (OB, FVG, Confluence,
    # Inducement...) مش موجودة في القمة، معناها الإضافات دي مش بتفيد
    # فعلياً للفريم ده رغم وجودها، والاعتماد الحقيقي على مؤشرات فنية
    # عامة (ATR, EMA) بس. ───
    importances = prod_model.feature_importances_
    feat_imp_pairs = sorted(zip(fe_prod.feature_columns, importances),
                             key=lambda x: x[1], reverse=True)
    top_20 = feat_imp_pairs[:20]
    smc_keywords = ("ob_", "fvg", "confluence", "inducement", "liquidity", "breaker",
                     "mitigat", "displacement", "ote", "choch", "bos_", "premium",
                     "discount", "session_momentum", "killzone", "kz_")
    smc_importance_share = sum(
        imp for feat, imp in feat_imp_pairs
        if any(k in feat.lower() for k in smc_keywords)
    )
    logging.info(f"[{tf_key}] Top 10 Feature Importance: " +
                 ", ".join(f"{f}={round(float(i),3)}" for f, i in top_20[:10]))
    logging.info(f"[{tf_key}] نسبة أهمية فيتشورز SMC من الإجمالي: "
                 f"{smc_importance_share*100:.1f}%")

    model_path = os.path.join(MODELS_DIR, f"{SYMBOL_SAFE}_{tf_key}_smc_gbpusd_v3.pkl")
    joblib.dump({
        "model": prod_model, "scaler": fe_prod.scaler,
        "feature_columns": fe_prod.feature_columns, "config": calibrated_config,
        "note": "trained on 100% of data (incl. holdout) — for live deployment only, "
                "holdout metrics below do NOT apply to this exact model. "
                "buy_threshold/sell_threshold in this config are auto-calibrated "
                "from a held-out calibration slice, not fixed guesses.",
    }, model_path)

    metrics_path = os.path.join(MODELS_DIR, f"{SYMBOL_SAFE}_{tf_key}_smc_gbpusd_v3_metrics.json")
    with open(metrics_path, "w") as f:
        json.dump({
            "timeframe": tf_key,
            "fold_results": fold_metrics,
            "holdout_backtest_true_oos": holdout_bt,
            "holdout_note": "evaluated with eval_model, trained ONLY on the pre-holdout "
                             "train_pool — never saw holdout data. This is the honest metric.",
            "config_static": config,
            "config_calibrated": calibrated_config,
            "n_features": len(fe.feature_columns),
            "top_20_feature_importance": [
                {"feature": f, "importance": round(float(i), 4)} for f, i in top_20
            ],
            "smc_features_importance_share_pct": round(smc_importance_share * 100, 1),
        }, f, indent=2, default=str)

    logging.info(f"تم حفظ نموذج الإنتاج: {model_path}")
    viable = calibrated_config.get("viable_for_standalone_trading", True)
    return prod_model, fe_prod, holdout_bt, viable, effective_mode


# ══════════════════════════════════════════════════════════════
# TRAIN ALL TIMEFRAMES
# ══════════════════════════════════════════════════════════════
def _holdout_truly_profitable(holdout_bt, min_pf=1.05):
    """
    فحص متدرج للأداء على شريحة الـ holdout الحقيقي (بيانات لم تُرَ نهائياً
    أثناء التدريب أو المعايرة).

    قبول متدرج بيراعي عدد الصفقات (مش عتبة قاطعة PF>1.05 فقط):
      - PF >= 1.15 + عدد صفقات >= 10 → قبول واضح (edge معقول)
      - PF >= 1.05 + عدد صفقات >= 20 → قبول (edge ضعيف لكن مستقر إحصائياً)
      - PF >= 1.00 + عدد صفقات >= 40 → قبول حذر (لا خسارة + عينة كبيرة)
      - غير كده → رفض (مش موثوق للتداول المستقل)

    المنطق: عينة كبيرة بـ PF متواضع أدل على edge حقيقي من عينة صغيرة
    بـ PF عالي (اللي ممكن يكون صدفة إحصائية).
    """
    if not holdout_bt:
        return False
    pf = holdout_bt.get("profit_factor", 0)
    n = holdout_bt.get("total_trades", 0)
    if pf >= 1.15 and n >= 10:
        return True
    if pf >= 1.05 and n >= 20:
        return True
    if pf >= 1.00 and n >= 40:
        return True
    return False


def train_smc_model_auto(tf_key, n_splits=5, mtf_builder=None, force_refresh=False, gsr_provider=None):
    """
    يجرب triple-barrier الأول (الطريقة الأدق منهجياً). لو الفريم عدّى
    بوابة المعايرة لكن فشل على الـ holdout الحقيقي (overfitting)، أو
    ما لقاش عتبة مربحة أصلاً، بيجرب طريقة الهدف البديلة (atr_filter).
    """
    result = train_smc_model_for_timeframe(
        tf_key, n_splits=n_splits, mtf_builder=mtf_builder,
        force_refresh=force_refresh, target_mode="barrier", gsr_provider=None,
    )
    if result is None:
        return None
    prod_model, fe_prod, holdout_bt, viable, mode_used = result
    holdout_ok = _holdout_truly_profitable(holdout_bt)
    if viable and holdout_ok:
        return prod_model, fe_prod, holdout_bt

    if viable and not holdout_ok:
        _pf = holdout_bt.get('profit_factor', 0) if holdout_bt else 0
        _n = holdout_bt.get('total_trades', 0) if holdout_bt else 0
        logging.warning(
            f"[{tf_key}] ⚠️ triple-barrier عدّى بوابة المعايرة (viable=True) لكن "
            f"الـ holdout الحقيقي طلع PF={_pf} على {_n} صفقة — مش كافي لبوابة "
            f"القبول المتدرجة (محتاجين PF≥1.15 مع 10+ صفقات، أو PF≥1.05 مع 20+، "
            f"أو PF≥1.00 مع 40+). جاري تجربة الطريقة البديلة (atr_filter)..."
        )
    else:
        logging.warning(f"[{tf_key}] طريقة triple-barrier مالقتش عتبة مربحة — "
                         f"جاري إعادة المحاولة بطريقة فلترة ATR البسيطة كبديل...")

    result2 = train_smc_model_for_timeframe(
        tf_key, n_splits=n_splits, mtf_builder=mtf_builder,
        force_refresh=False, target_mode="atr_filter", gsr_provider=None,
    )
    if result2 is None:
        logging.warning(f"[{tf_key}] الطريقة البديلة فشلت كمان — هنفضل بموديل "
                         f"triple-barrier (غير مضمون الربحية، لكن الوحيد المتاح)")
        return prod_model, fe_prod, holdout_bt

    prod_model2, fe_prod2, holdout_bt2, viable2, mode_used2 = result2
    holdout2_ok = _holdout_truly_profitable(holdout_bt2)

    if viable2 and holdout2_ok:
        logging.info(f"✅ [{tf_key}] الطريقة البديلة (atr_filter) عدّت المعايرة "
                      f"*و* الـ holdout الحقيقي — هنستخدمها")
        return prod_model2, fe_prod2, holdout_bt2

    def _score(bt):
        if not bt or not bt.get("total_trades"):
            return -np.inf
        return bt.get("profit_factor", 0) * np.log1p(bt["total_trades"])

    score1, score2 = _score(holdout_bt), _score(holdout_bt2)
    if score2 > score1:
        logging.warning(
            f"[{tf_key}] ولا طريقة عمّمت فعلياً للـ holdout — atr_filter أفضل قليلاً "
            f"(holdout PF={holdout_bt2.get('profit_factor', 0) if holdout_bt2 else 0}) "
            f"من triple-barrier (PF={holdout_bt.get('profit_factor', 0) if holdout_bt else 0}). "
            f"هنحفظه، لكنه *غير موثوق* للتداول المستقل حالياً — استخدمه كـ HTF "
            f"context/bias بس لحد ما يبان edge أوضح ببيانات/فيتشورز إضافية."
        )
        return prod_model2, fe_prod2, holdout_bt2
    else:
        logging.warning(
            f"[{tf_key}] ولا الطريقتين عمّمتا فعلياً للـ holdout — هنفضل بـ "
            f"triple-barrier (PF={holdout_bt.get('profit_factor', 0) if holdout_bt else 0}) "
            f"كأفضل الموجود، لكنه *غير موثوق* للتداول المستقل حالياً."
        )
        return prod_model, fe_prod, holdout_bt


def train_all_smc_models(force_refresh_data=False):
    logging.info("=" * 70)
    logging.info("SMC GBP/USD Model Factory")
    logging.info("=" * 70)

    mtf_builder = MultiTimeframeBuilder()
    gsr_provider = None  # GBP/USD لا يحتاج نسبة الدهب/الفضة
    results = {}
    for tf in TIMEFRAME_CONFIG:
        res = train_smc_model_auto(
            tf, n_splits=5, mtf_builder=mtf_builder, force_refresh=force_refresh_data,
            gsr_provider=None,
        )
        results[tf] = "✅" if res else "❌"

    logging.info("=" * 70)
    logging.info("ملخص التدريب:")
    for tf, st in results.items():
        logging.info(f"   {tf}: {st}")
    logging.info("=" * 70)


if __name__ == "__main__":
    train_all_smc_models(force_refresh_data=False)

In [10]:
import os, json, glob
import pandas as pd

MODELS_DIR = "models/smc_gbpusd_v2"
SYMBOL = "GBPUSD"

files = sorted(glob.glob(os.path.join(MODELS_DIR, f"{SYMBOL}_*_smc_gbpusd_v2_metrics.json")))
results = {}
for f in files:
    with open(f, encoding="utf-8") as fp:
        data = json.load(fp)
    results[data.get("timeframe")] = data

order = ["5min", "15min", "45min", "1h", "2h", "4h", "1d", "1W"]
rows = []
for tf in order:
    if tf not in results: continue
    bt = results[tf].get("holdout_backtest_true_oos") or {}
    cfg = results[tf].get("config_calibrated") or {}
    rows.append({
        "الفريم": tf,
        "الصفقات": bt.get("total_trades", 0),
        "Win Rate %": bt.get("win_rate", 0),
        "Profit Factor": bt.get("profit_factor", 0),
        "العائد %": bt.get("total_return_pct", 0),
        "أقصى انخفاض %": bt.get("max_drawdown_pct", 0),
        "موثوق؟": "✅" if bt.get("statistically_reliable") else "⚠️",
        "Viable؟": "✅" if cfg.get("viable_for_standalone_trading") else "❌",
        "Buy Th": round(cfg.get("buy_threshold", 0), 3),
        "Sell Th": round(cfg.get("sell_threshold", 0), 3),
    })

print(pd.DataFrame(rows).to_string(index=False))

الفريم  الصفقات  Win Rate %  Profit Factor  العائد %  أقصى انخفاض % موثوق؟ Viable؟  Buy Th  Sell Th
  5min      314       29.62           1.74      0.29          -0.02      ✅       ✅   0.619    0.317
 15min      134       53.73           2.68      0.68          -0.05      ✅       ✅   0.659    0.158
 45min       96       35.42           0.94     -0.04          -0.16      ✅       ✅   0.608    0.398
    1h      512       38.28           0.93     -0.28          -0.57      ✅       ✅   0.508    0.490
    2h        9       44.44           0.64     -0.05          -0.12     ⚠️       ✅   0.521    0.449
    4h      169       48.52           1.29      0.72          -0.32      ✅       ✅   0.510    0.492
    1d       39       43.59           0.98     -0.06          -0.82      ✅       ❌   0.550    0.400
    1W        9       77.78           4.18      1.66          -0.25      ✅       ❌   0.530    0.450


## تحميل الموديلات
شغّل الخلية دي بعد انتهاء التدريب فوق عشان تنزّل كل الموديلات كملف ZIP.

In [11]:
# ============================================================
# download_models
import os
import shutil
from google.colab import files

MODELS_DIR = "models/smc_gbpusd"
ZIP_NAME   = "smc_gbpusd.zip"

if not os.path.exists(MODELS_DIR):
    print(f"❌ الفولدر '{MODELS_DIR}' مش موجود!")
    print("تأكد إن الخلية اللي فوق اشتغلت وحفظت الملفات أولاً.")
else:
    all_files = os.listdir(MODELS_DIR)
    pkl_files  = [f for f in all_files if f.endswith('.pkl')]
    json_files = [f for f in all_files if f.endswith('.json')]

    print(f"📁 محتوى '{MODELS_DIR}':")
    print(f"   .pkl  (موديلات): {len(pkl_files)}")
    print(f"   .json (مقاييس):  {len(json_files)}")
    print()
    for f in sorted(all_files):
        size = os.path.getsize(os.path.join(MODELS_DIR, f))
        print(f"   {f:<45} {size/1024:.1f} KB")

    print(f"\n📦 جاري ضغط الفولدر ...")
    shutil.make_archive(ZIP_NAME.replace('.zip', ''), 'zip', '.', MODELS_DIR)

    size_mb = os.path.getsize(ZIP_NAME) / (1024 * 1024)
    print(f"✅ تم إنشاء: {ZIP_NAME} ({size_mb:.1f} MB)")

    print(f"⬇️  جاري التحميل ...")
    files.download(ZIP_NAME)
    print(f"✅ تم!")


❌ الفولدر 'models/smc_gbpusd' مش موجود!
تأكد إن الخلية اللي فوق اشتغلت وحفظت الملفات أولاً.
